In [1]:
import os
import re
import math
import json
import logging
import random
import numpy as np
import pandas as pd
import scipy.linalg
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from typing import Optional
from torchvision.models import inception_v3, Inception_V3_Weights
from tqdm import tqdm
from scipy.linalg import sqrtm
import operator
from scipy.optimize import linear_sum_assignment
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [2]:
# -------------------------------
# 設定 logging 等級，方便印出訓練與評估時的訊息
# -------------------------------
logging.basicConfig(level=logging.INFO)

# -------------------------------
# 常數定義：時間嵌入維度
# -------------------------------
TIME_EMB_DIM = 128

# --------------------------------------
# 數據處理相關
# --------------------------------------
def parse_lat_lon(column_name: str) -> tuple[float, float]:
    """
    解析欄位名稱中的經緯度資訊，假設格式為 "name (lon, lat)"。

    參數:
        column_name: 欄位名稱，必須包含以括號包住的經緯度資訊，例如 "(121.565, 25.033)"。

    回傳:
        一個元組 (經度, 緯度) 的浮點數。

    若格式不正確則拋出 ValueError。
    """
    match = re.search(r'\(([\d.-]+),\s*([\d.-]+)\)', column_name)
    if match:
        return float(match.group(1)), float(match.group(2))
    raise ValueError(f"欄位名稱格式無效：{column_name}")

class PeopleFlowDatasetCondition(Dataset):
    def __init__(self, csv_path: str, H: int, W: int, condition_length: int, 
                 prediction_length: int, transform: Optional[callable] = None, 
                 normalize: bool = True, debug: bool = False):
        # 若 CSV 檔案不存在則拋出錯誤
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"CSV 檔案未找到：{csv_path}")
        
        # 讀取 CSV 數據
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.condition_length = condition_length
        self.prediction_length = prediction_length
        self.total_length = condition_length + prediction_length
        self.normalize = normalize
        self.H, self.W = H, W

        # 處理額外條件欄位
        extra_cols_list = [
            "測站氣壓", "海平面氣壓", "氣溫", "露點溫度", "相對溼度", "風速", "最大陣風",
            "降水量", "降水時數", "日照時數", "全天空日射量", "能見度", "紫外線指數", "總雲量",
            "holiday", "weekday", "年", "月", "日", "時",
            "sin_風向", "cos_風向", "sin_最大陣風風向", "cos_最大陣風風向"
        ]
        if "hoilday" in self.df.columns:
            self.df.rename(columns={"hoilday": "holiday"}, inplace=True)
        
        # 風向轉換為 sin 和 cos
        for col in ['最大陣風風向', '風向']:
            if col in self.df.columns:
                self.df[f'sin_{col}'] = np.sin(np.deg2rad(self.df[col]))
                self.df[f'cos_{col}'] = np.cos(np.deg2rad(self.df[col]))
                self.df.drop(columns=[col], inplace=True)
        
        df_extra = self.df[extra_cols_list].copy()
        cat_features = ['holiday']
        df_extra[cat_features] = df_extra[cat_features].astype(str)
        df_cat = pd.get_dummies(df_extra[cat_features], prefix=cat_features)
        df_cont = df_extra.drop(columns=cat_features)
        
        if normalize:
            cont_mean = df_cont.mean()
            cont_std = df_cont.std() + 1e-5
            df_cont = (df_cont - cont_mean) / cont_std
            self.extra_cont_mean = cont_mean
            self.extra_cont_std = cont_std
        
        df_extra_processed = pd.concat([df_cont, df_cat], axis=1)
        self.extra_columns = list(df_extra_processed.columns)
        self.extra_data = df_extra_processed.values.astype(np.float32)

         # 1. 提取所有可用的經緯度座標及其原始欄位名
        all_flow_columns_with_coords = [c for c in self.df.columns if '(' in c and ')' in c]
        
        num_required_points = H * W
        if len(all_flow_columns_with_coords) < num_required_points:
            raise ValueError(
                f"網格大小 ({H}x{W}={num_required_points}) 大於了可用的地理座標點數量 ({len(all_flow_columns_with_coords)})."
                " 請減少 H*W 或提供更多座標點。"
            )
        
        # 解析所有座標點
        all_column_info = [] # 存儲 (原始欄位名, lon, lat, 原始索引)
        for original_idx, col_name in enumerate(all_flow_columns_with_coords):
            lon, lat = parse_lat_lon(col_name)
            all_column_info.append({'name': col_name, 'lon': lon, 'lat': lat, 'original_idx': original_idx})
        
        all_coords_np = np.array([(info['lon'], info['lat']) for info in all_column_info])

        # 如果座標點多於網格數，選擇最靠近幾何中心的 num_required_points 個點
        if len(all_column_info) > num_required_points:
            print(f"訊息: 座標點數量 ({len(all_column_info)}) 多於網格數 ({num_required_points}). "
                  f"將選擇最靠近地理中心的 {num_required_points} 個座標點進行映射。")
            
            # 計算所有點的幾何中心
            geometric_center_lon = np.mean(all_coords_np[:, 0])
            geometric_center_lat = np.mean(all_coords_np[:, 1])
            
            # 計算每個點到幾何中心的距離
            distances_to_geometric_center = np.sqrt(
                (all_coords_np[:, 0] - geometric_center_lon)**2 +
                (all_coords_np[:, 1] - geometric_center_lat)**2
            )
            
            # 獲取距離最近的 num_required_points 個點的索引 (相對於 all_coords_np)
            selected_indices_in_all_coords = np.argsort(distances_to_geometric_center)[:num_required_points]
            
            # 更新 self.column_info 和 real_coords_np 只包含選中的點
            self.column_info = [all_column_info[i] for i in selected_indices_in_all_coords]
            real_coords_np = all_coords_np[selected_indices_in_all_coords]
            
        elif len(all_column_info) == num_required_points:
            self.column_info = all_column_info
            real_coords_np = all_coords_np
        else: 
            pass 


        # --- 後續的匈牙利算法分配邏輯與之前相同，使用 self.column_info 和 real_coords_np ---
        # 2. 計算網格的理論目標地理中心
        overall_mean_lon, overall_mean_lat = np.mean(real_coords_np, axis=0)
        grid_center_lon, grid_center_lat = overall_mean_lon, overall_mean_lat

        unique_lons = np.unique(real_coords_np[:, 0])
        unique_lats = np.unique(real_coords_np[:, 1])
        lon_diffs = np.diff(np.sort(unique_lons))
        lat_diffs = np.diff(np.sort(unique_lats))
        
        lon_step = np.median(lon_diffs[lon_diffs > 0]) if len(lon_diffs[lon_diffs > 0]) > 0 else 0.005
        lat_step = np.median(lat_diffs[lat_diffs > 0]) if len(lat_diffs[lat_diffs > 0]) > 0 else 0.005
        if lon_step == 0: lon_step = 0.005
        if lat_step == 0: lat_step = 0.005

        grid_target_coords = np.zeros((num_required_points, 2))
        grid_idx_to_rc_map = {}
        current_grid_idx = 0
        for r in range(H):
            for c in range(W):
                target_lon = grid_center_lon + (c - (W - 1) / 2.0) * lon_step
                target_lat = grid_center_lat - (r - (H - 1) / 2.0) * lat_step
                grid_target_coords[current_grid_idx, 0] = target_lon
                grid_target_coords[current_grid_idx, 1] = target_lat
                grid_idx_to_rc_map[current_grid_idx] = (r, c)
                current_grid_idx += 1
        
        # 3. 計算成本矩陣
        cost_matrix = np.zeros((num_required_points, num_required_points))
        for i in range(num_required_points):
            for j in range(num_required_points):
                dist_sq = (real_coords_np[i, 0] - grid_target_coords[j, 0])**2 + \
                          (real_coords_np[i, 1] - grid_target_coords[j, 1])**2
                cost_matrix[i, j] = np.sqrt(dist_sq)

        # 4. 使用匈牙利算法
        assigned_real_coord_indices, assigned_grid_indices = linear_sum_assignment(cost_matrix)
        
        # 5. 構建最終的網格
        final_grid_assignment = np.full((H, W), -1, dtype=int) # 存儲 real_coords_np 中的索引
        
        temp_assignment_map = {grid_idx: real_idx for real_idx, grid_idx in zip(assigned_real_coord_indices, assigned_grid_indices)}
        
        flat_grid_indices_in_order = [] # 按照 (0,0)...(H-1,W-1) 順序排列的、分配到這些網格的 real_coords_np 索引

        for grid_1d_idx in range(num_required_points):
            r_map, c_map = grid_idx_to_rc_map[grid_1d_idx]
            # real_coord_idx_for_this_grid 是 temp_assignment_map 的 value, 它是 real_coords_np 的索引
            real_coord_idx_for_this_grid = temp_assignment_map[grid_1d_idx]
            final_grid_assignment[r_map, c_map] = real_coord_idx_for_this_grid # 這裡存的是 real_coords_np 的索引
            flat_grid_indices_in_order.append(real_coord_idx_for_this_grid)

        # self.sorted_flow_columns 應使用 self.column_info (它現在只包含選中的點)
        # flat_grid_indices_in_order 中的索引是相對於 real_coords_np 和 self.column_info 的
        self.sorted_flow_columns = [self.column_info[idx]['name'] for idx in flat_grid_indices_in_order]


        self._plot_grid(save_path=r"C:\thesis\code\result_ddpm\plot_grid_hungarian_centered_selection.png")

        # 6. 根據排序好的欄位順序取出 CSV 中的數據
        flow_values = self.df[self.sorted_flow_columns].values.reshape(-1, H, W).astype(np.float32)
        self.data = torch.from_numpy(flow_values)
        
        if normalize:
            self.mean_val = self.data.mean()
            self.std_val = self.data.std() + 1e-5
            self.data = (self.data - self.mean_val) / self.std_val
        
        self.max_index = self.data.shape[0] - self.total_length + 1

    def _plot_grid(self, save_path: str):
        """
        繪製網格圖，顯示每個網格點對應的經緯度及其座標位置，
        並將結果存檔到指定的路徑。
        """
        locations = [parse_lat_lon(col) for col in self.sorted_flow_columns]
        longitudes, latitudes = zip(*locations)
        plt.figure(figsize=(12, 12))
        plt.scatter(longitudes, latitudes, c='blue', marker='o', label='Grid Points')
        # 在每個點旁顯示網格索引
        for i in range(self.H):
            for j in range(self.W):
                idx = i * self.W + j
                plt.text(longitudes[idx], latitudes[idx], f'[{i},{j}]', fontsize=6, ha='right')
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title("Grid Arrangement")
        plt.grid(True)
        plt.legend()
        plt.savefig(save_path, dpi=600, bbox_inches='tight', pad_inches=0.1)
        plt.close()

    def __len__(self) -> int:
        # 數據集的總樣本數為可滑動視窗的數量
        return self.max_index

    def __getitem__(self, idx):
        cond_seq = self.data[idx:idx + self.condition_length]  # (8, 21, 21)
        target_seq = self.data[idx + self.condition_length:idx + self.total_length]  # (1, 21, 21)
        model_input = torch.cat([cond_seq, target_seq], dim=0).unsqueeze(0)  # (1, 9, 21, 21)
        target_seq = target_seq.unsqueeze(0)  # (1, 1, 21, 21)
        extra_data = torch.tensor(self.extra_data[idx:idx + self.condition_length], dtype=torch.float32)
        return model_input, target_seq, extra_data

def collate_fn(batch):
    model_inputs, targets, extras = zip(*batch)
    model_inputs = torch.stack(model_inputs)  # (batch_size, 1, 9, 21, 21)
    targets = torch.stack(targets)  # (batch_size, 1, 1, 21, 21)
    extras = torch.stack(extras)  # (batch_size, extra_dim)
    return model_inputs, targets, extras

In [3]:
# --------------------------------------
# 模型定義
# --------------------------------------
class DoubleConv3D(nn.Module):
    """
    定義 3D 卷積層組合，包含兩次卷積、BatchNorm 與 ReLU 激活函數。
    此結構常用於 U-Net 中作為基本模組。
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Sequential(
            # 第一次卷積
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            # 第二次卷積
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class UNet3D(nn.Module):
    """
    3D U-Net 結構，包含下採樣（Encoder）、中間瓶頸層與上採樣（Decoder）。
    此模型同時接收噪聲版本的目標數據 x_t、完整序列 x_full 與時間嵌入。
    """
    def __init__(self, in_channels=1, base_channels=64, time_emb_dim=128, dropout_rate=0.0):
        super().__init__()
        # 編碼器部分：逐層進行雙卷積與下採樣
        self.enc1 = DoubleConv3D(in_channels, base_channels)
        self.pool1 = nn.MaxPool3d((2, 2, 2))
        self.enc2 = DoubleConv3D(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool3d((2, 2, 2))
        self.enc3 = DoubleConv3D(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool3d((2, 2, 2))
        self.enc4 = DoubleConv3D(base_channels * 4, base_channels * 8)
        # 這裡使用不同的池化參數以調整深度與空間尺寸
        self.pool4 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(1, 2, 2), padding=(1, 0, 0))
        # 瓶頸層
        self.bottleneck = DoubleConv3D(base_channels * 8, base_channels * 16)
        # 解碼器部分：逐層上採樣並與對應編碼層做 concat
        self.up4 = nn.ConvTranspose3d(base_channels * 16, base_channels * 8, kernel_size=(2, 2, 2), stride=(1, 2, 2), output_padding=(0, 1, 1))
        self.dec4 = DoubleConv3D(base_channels * 16, base_channels * 8)
        self.up3 = nn.ConvTranspose3d(base_channels * 8, base_channels * 4, kernel_size=(2, 2, 2), stride=(2, 2, 2), output_padding=(1, 0, 0))
        self.dec3 = DoubleConv3D(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose3d(base_channels * 4, base_channels * 2, kernel_size=(2, 2, 2), stride=(2, 2, 2))
        self.dec2 = DoubleConv3D(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose3d(base_channels * 2, base_channels, kernel_size=(2, 2, 2), stride=(2, 2, 2))
        self.dec1 = DoubleConv3D(base_channels * 2, base_channels)
        # 輸出卷積，將通道數降為 1
        self.out_conv = nn.Conv3d(base_channels, 1, kernel_size=1)
        # dropout 用於防止過擬合
        self.dropout = nn.Dropout3d(dropout_rate)
        # 時間嵌入的線性轉換與激活
        self.time_proj = nn.Sequential(nn.Linear(time_emb_dim, base_channels * 8), nn.SiLU())
        # 將完整序列 x_full 通過 1x1 卷積調整通道數，使其與 x_t 保持一致（這裡假設保持 1 通道）
        self.x_full_conv = nn.Conv3d(in_channels, in_channels, kernel_size=1)

    def forward(self, x_t, x_full, t_emb):
        """
        前向傳播函數：
        參數:
            x_t: 含噪聲的部分序列，形狀 (batch, 1, 9, 21, 21)
            x_full: 完整的序列數據，形狀 (batch, 1, 9, 21, 21)
            t_emb: 時間嵌入，形狀 (batch, time_emb_dim)
        回傳:
            模型輸出，形狀 (batch, 1, 1, 21, 21)
        """
        # 處理 x_full，使其通道數與 x_t 一致
        x_full_conv = self.x_full_conv(x_full)  # (batch, 1, 9, 21, 21)
        # 將 x_t 與處理後的 x_full 做融合（逐元素相加）
        x_input = x_t + x_full_conv  # (batch, 1, 9, 21, 21)
        # 編碼器第一層
        e1 = self.enc1(x_input)  # (batch, 64, 9, 21, 21)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        p4 = self.pool4(e4)
        # 將時間嵌入經線性轉換後擴展至與 p4 同維度並與 p4 相加
        t_emb = self.time_proj(t_emb)[:, :, None, None, None]
        b = self.bottleneck(p4 + t_emb)
        b = self.dropout(b)
        # 解碼器：上採樣後與對應編碼層做 concat
        d4 = self.up4(b)
        if d4.shape[-3:] != e4.shape[-3:]:
            # 使用 trilinear 插值調整尺寸
            d4 = F.interpolate(d4, size=e4.shape[-3:], mode='trilinear', align_corners=True)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        d3 = self.up3(d4)
        if d3.shape[-3:] != e3.shape[-3:]:
            d3 = F.interpolate(d3, size=e3.shape[-3:], mode='trilinear', align_corners=True)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        if d2.shape[-3:] != e2.shape[-3:]:
            d2 = F.interpolate(d2, size=e2.shape[-3:], mode='trilinear', align_corners=True)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        if d1.shape[-3:] != e1.shape[-3:]:
            d1 = F.interpolate(d1, size=e1.shape[-3:], mode='trilinear', align_corners=True)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        # 經過輸出卷積獲得最終結果
        out = self.out_conv(d1)
        # 返回結果，僅保留時間維度上的第一個步驟（1個預測步長）
        return out[:, :, :1, :, :]

class DDPM3D(nn.Module):
    """
    條件式 DDPM (Denoising Diffusion Probabilistic Model) 模型。
    此模型利用前向擴散與反向去噪過程進行生成任務。
    """
    def __init__(self, model: nn.Module, timesteps: int = 1000, 
                 beta_start: float = 1e-4, beta_end: float = 0.02, device: str = 'cuda'):
        super().__init__()
        self.model = model
        self.timesteps = timesteps
        self.device = device
        # 線性生成 beta 值
        self.betas = torch.linspace(beta_start, beta_end, timesteps).to(device)
        self.alphas = 1.0 - self.betas
        # 累乘計算 alpha 的連乘積，用於生成擴散過程的係數
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        # 計算頻率因子，用於時間嵌入的正弦與餘弦函數
        self.half_dim = TIME_EMB_DIM // 2
        self.freq_factor = torch.exp(torch.arange(self.half_dim, dtype=torch.float32) *
                                     -(math.log(10000.0) / (self.half_dim - 1))).to(device)

    def get_time_embedding(self, t):
        """
        生成時間嵌入向量，使用正弦與餘弦函數將標量時間映射到向量。
        參數:
            t: 時間步（batch_size,)
        回傳:
            時間嵌入，形狀 (batch_size, TIME_EMB_DIM)
        """
        t = t.float()
        emb = t[:, None] * self.freq_factor.to(t.device)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)

    def get_condition_embedding(self, cond):
        """
        取得條件嵌入，目前未使用條件資訊，返回全零向量。
        """
        return torch.zeros(cond.shape[0], TIME_EMB_DIM, device=cond.device)

    def q_sample(self, x0, t, noise=None):
        """
        前向擴散過程：根據給定的時間步 t，將數據 x0 擴散成含噪版本。
        參數:
            x0: 原始數據
            t: 時間步（batch_size,)
            noise: 可選噪聲，若未提供則生成隨機噪聲
        回傳:
            擴散後的數據
        """
        if noise is None:
            noise = torch.randn_like(x0)
        # 根據時間步獲取相對應的係數
        sqrt_alpha = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
        sqrt_one_minus_alpha = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
        # 混合原始數據與噪聲
        return sqrt_alpha * x0 + sqrt_one_minus_alpha * noise

    def p_losses(self, cond, target, t):
        """
        計算去噪損失，目標是讓模型預測出噪聲部分。
        參數:
            cond: 條件序列（含目標數據與前置條件）
            target: 真實目標數據
            t: 時間步
        回傳:
            均方誤差損失
        """
        # 構造完整無噪聲序列：將 target 與條件序列的後半部拼接
        x_full = torch.cat([target, cond[:, :, 1:]], dim=2)
        noise = torch.randn_like(target)
        # 生成含噪 target
        x_noisy_target = self.q_sample(target, t, noise=noise)
        # 將含噪 target 與條件序列拼接，作為模型輸入
        x_t = torch.cat([x_noisy_target, cond[:, :, 1:]], dim=2)
        # 取得時間嵌入與條件嵌入，並融合
        time_emb = self.get_time_embedding(t).to(self.device)
        cond_emb = self.get_condition_embedding(cond)
        combined_emb = time_emb + cond_emb
        # 預測噪聲
        pred_noise = self.model(x_t, x_full, combined_emb)
        pred_noise_target = pred_noise[:, :, :1, :, :]
        # 計算模型預測與實際噪聲間的均方誤差
        return F.mse_loss(pred_noise_target, noise)

    @torch.no_grad()
    def p_sample(self, x_t, t, cond):
        """
        單步反向去噪：從含噪數據 x_t 生成前一步的數據。
        參數:
            x_t: 當前含噪數據，形狀 (batch, 1, 1, H, W) 或 (batch, 1, 9, 21, 21)
            t: 當前時間步
            cond: 條件數據
        回傳:
            去噪後的數據 x_{t-1}
        """
        # 確保 x_t 與 cond 為 5 維張量
        if x_t.dim() == 4:
            x_t = x_t.unsqueeze(1)  # (batch, 1, 1, H, W)
        if cond.dim() == 4:
            cond = cond.unsqueeze(1)  # (batch, 1, 9, 21, 21)
        
        # 取得當前時間步的 beta 等係數
        beta_t = self.betas[t].view(-1, 1, 1, 1, 1)
        sqrt_recip_alpha_t = 1.0 / torch.sqrt(self.alphas[t]).view(-1, 1, 1, 1, 1)
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
        time_emb = self.get_time_embedding(t).to(self.device)
        cond_emb = self.get_condition_embedding(cond)
        combined_emb = time_emb + cond_emb
        
        # 將含噪數據與條件數據合併
        x_t_full = torch.cat([x_t, cond[:, :, :8]], dim=2)  # (batch_size, 1, 9, 21, 21)
        x_full = cond  # 完整序列作為條件
        # 模型預測噪聲
        eps_theta = self.model(x_t_full, x_full, combined_emb)
        eps_theta_target = eps_theta[:, :, :1, :, :]  # 僅取出目標噪聲部分
        
        # 反向去噪步驟：計算 x_{t-1}
        x_t_minus_1 = sqrt_recip_alpha_t * (x_t - beta_t / sqrt_one_minus_alphas_cumprod_t * eps_theta_target)
        # 當 t > 0 時，加入隨機噪聲；t = 0 時直接返回
        mask = (t > 0).float().view(-1, 1, 1, 1, 1)
        sigma_t = torch.sqrt(beta_t)
        noise = torch.randn_like(x_t)
        return x_t_minus_1 + mask * sigma_t * noise

    @torch.no_grad()
    def p_sample_loop(self, shape, cond):
        """
        反向去噪迴圈：從初始純噪聲開始，逐步去噪生成數據。
        參數:
            shape: 生成數據的形狀（可能需要調整為 5D）
            cond: 條件數據
        回傳:
            生成的數據張量
        """
        # 確保 cond 為 5D
        if cond.dim() == 4:
            cond = cond.unsqueeze(1)  # (batch, 1, 9, 21, 21)
        
        # 調整 shape 為 5D
        if len(shape) == 4:
            batch_size = shape[0]
            shape = (batch_size, 1, shape[1], shape[2], shape[3])
        
        # 初始噪聲
        x = torch.randn(shape, device=self.device)
        
        # 由最後一步開始，逐步進行去噪
        for i in reversed(range(self.timesteps)):
            t = torch.full((shape[0],), i, device=self.device, dtype=torch.long)
            x = self.p_sample(x, t, cond)
        
        return x

In [4]:
# --------------------------------------
# 視覺化工具：用於繪製預測結果、誤差網格圖等
# --------------------------------------
def truncate_colormap(cmap, minval: float = 0.0, maxval: float = 1.0, n: int = 256):
    """
    截斷 colormap，僅使用其中一部分的色階範圍。
    參數:
        cmap: 原始的 colormap
        minval, maxval: 取色範圍
        n: 取樣點數
    回傳:
        新的截斷後的 colormap
    """
    new_cmap = mcolors.LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n))
    )
    return new_cmap

def visualize_predictions(cond, generated, target, sample_idx: int = 0, 
                         save_dir: str = r"C:\thesis\code\result_ddpm_hierarchical"):
    """
    視覺化預測結果與真實值的比較，包含生成結果、真實數據、以及誤差（MSE 與 MAE）的圖形。
    參數:
        cond: 條件數據
        generated: 生成結果
        target: 真實目標數據
        sample_idx: 指定要視覺化哪個樣本
        save_dir: 圖形存檔的目錄
    """
    os.makedirs(save_dir, exist_ok=True)
    pred_length = generated.shape[2]
    
    if sample_idx is None:
        generated_avg = torch.mean(generated, dim=(0, 2)).squeeze(0).cpu().numpy()
        target_avg = torch.mean(target, dim=(0, 2)).squeeze(0).cpu().numpy()
        mse_matrix = (generated_avg - target_avg) ** 2
        mae_matrix = np.abs(generated_avg - target_avg)
        mape_matrix = np.abs((target_avg - generated_avg) / (target_avg + 1e-10)) * 100
        smape_matrix = np.abs(generated_avg - target_avg) / (np.abs(target_avg) + np.abs(generated_avg) + 1e-10) * 100
        
        mse = np.mean(mse_matrix)
        mae = np.mean(mae_matrix)
        mape = np.mean(mape_matrix)
        smape = np.mean(smape_matrix)
        
        plt.figure(figsize=(6, 6))
        plt.imshow(generated_avg, cmap='viridis')
        plt.colorbar()
        plt.title('avg_generated')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_generated.png'), dpi=300)
        plt.close()
        
        plt.figure(figsize=(6, 6))
        plt.imshow(target_avg, cmap='viridis')
        plt.colorbar()
        plt.title('avg_target')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_target.png'), dpi=300)
        plt.close()
        
        plt.figure(figsize=(6, 6))
        plt.imshow(mse_matrix, cmap='hot')
        plt.colorbar()
        plt.title(f'MSE: {mse:.0f}')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_mse.png'), dpi=300)
        plt.close()
        
        plt.figure(figsize=(6, 6))
        plt.imshow(mae_matrix, cmap='hot')
        plt.colorbar()
        for i in range(mae_matrix.shape[0]):
            for j in range(mae_matrix.shape[1]):
                plt.text(j, i, f'{int(round(mae_matrix[i, j]))}', ha='center', va='center', color='white', fontsize=4)
        plt.title(f'MAE: {mae:.0f}')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_mae.png'), dpi=300)
        plt.close()
        
        plt.figure(figsize=(6, 6))
        plt.imshow(mape_matrix, cmap='hot')
        plt.colorbar()
        for i in range(mape_matrix.shape[0]):
            for j in range(mape_matrix.shape[1]):
                plt.text(j, i, f'{int(round(mape_matrix[i, j]))}', ha='center', va='center', color='white', fontsize=4)
        plt.title(f'MAPE: {mape:.0f}%')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_mape.png'), dpi=300)
        plt.close()
        
        plt.figure(figsize=(6, 6))
        plt.imshow(smape_matrix, cmap='hot')
        plt.colorbar()
        for i in range(smape_matrix.shape[0]):
            for j in range(smape_matrix.shape[1]):
                plt.text(j, i, f'{int(round(smape_matrix[i, j]))}', ha='center', va='center', color='white', fontsize=4)
        plt.title(f'SMAPE: {smape:.0f}%')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_smape.png'), dpi=300)
        plt.close()
    
    else:
        for t in range(pred_length):
            plt.figure(figsize=(20, 4))
            
            plt.subplot(1, 6, 1)
            plt.imshow(generated[sample_idx, 0, t].cpu().numpy(), cmap='viridis')
            plt.colorbar()
            plt.title(f'Generated (t={t})')
            
            plt.subplot(1, 6, 2)
            plt.imshow(target[sample_idx, 0, t].cpu().numpy(), cmap='viridis')
            plt.colorbar()
            plt.title(f'True (t={t})')
            
            error_sq = (generated[sample_idx, 0, t].cpu().numpy() - target[sample_idx, 0, t].cpu().numpy()) ** 2
            plt.subplot(1, 6, 3)
            plt.imshow(error_sq, cmap='hot')
            plt.colorbar()
            plt.title(f'MSE (t={t})')
            
            error_abs = np.abs(generated[sample_idx, 0, t].cpu().numpy() - target[sample_idx, 0, t].cpu().numpy())
            plt.subplot(1, 6, 4)
            plt.imshow(error_abs, cmap='hot')
            plt.colorbar()
            plt.title(f'MAE (t={t})')
            
            mape = np.abs((target[sample_idx, 0, t].cpu().numpy() - generated[sample_idx, 0, t].cpu().numpy()) / 
                         (target[sample_idx, 0, t].cpu().numpy() + 1e-10)) * 100
            plt.subplot(1, 6, 5)
            plt.imshow(mape, cmap='hot')
            plt.colorbar()
            plt.title(f'MAPE (t={t})')
            
            # 新增條件數據視覺化（最後一個條件時間步）
            if cond is not None:
                plt.subplot(1, 6, 6)
                plt.imshow(cond[sample_idx, 0, -1].cpu().numpy(), cmap='viridis')
                plt.colorbar()
                plt.title(f'Condition (t={cond.shape[2]-1})')
            
            plt.suptitle(f'Sample {sample_idx} - Time Step {t}')
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.savefig(os.path.join(save_dir, f'prediction_sample{sample_idx}_t{t}.png'), dpi=300)
            plt.close()

def plot_grid_with_error(sorted_flow_columns: list, H: int, W: int, 
                         mse_matrix: np.ndarray, mae_matrix: np.ndarray, mape_matrix: np.ndarray, 
                         save_dir: str = r"C:\\thesis\\code\\result_ddpm_hierarchical", smape_matrix: np.ndarray = None):
    """
    繪製網格圖，顯示每個網格點的誤差（MSE、MAE 和 MAPE），並將結果存成圖與表格。
    
    Args:
        sorted_flow_columns (list): 經緯度欄位的排序列表。
        H (int): 網格高度。
        W (int): 網格寬度。
        mse_matrix (np.ndarray): 每個網格點的 MSE 矩陣，形狀為 (H, W)。
        mae_matrix (np.ndarray): 每個網格點的 MAE 矩陣，形狀為 (H, W)。
        mape_matrix (np.ndarray): 每個網格點的 MAPE 矩陣，形狀為 (H, W)。
        save_dir (str): 存檔路徑。
        smape_matrix (np.ndarray, optional): 每個網格點的 SMAPE 矩陣，形狀為 (H, W)。預設為 None。
    """
    os.makedirs(save_dir, exist_ok=True)
    
    locations = [parse_lat_lon(col) for col in sorted_flow_columns]
    longitudes, latitudes = zip(*locations)
    
    orig_cmap = plt.get_cmap('OrRd')
    trunc_cmap = truncate_colormap(orig_cmap, 0.3, 1.0)
    
    # 繪製 MSE 網格圖
    plt.figure(figsize=(12, 12))
    scatter = plt.scatter(longitudes, latitudes, c=mse_matrix.flatten(), cmap=trunc_cmap, marker='o')
    plt.colorbar(scatter, label='MSE')
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid with MSE")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_mse.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

    # 繪製 MAE 網格圖
    plt.figure(figsize=(12, 12))
    scatter = plt.scatter(longitudes, latitudes, c=mae_matrix.flatten(), cmap=trunc_cmap, marker='o')
    plt.colorbar(scatter, label='MAE')
    for i, (lon, lat) in enumerate(zip(longitudes, latitudes)):
        plt.text(lon, lat, f'{int(round(mae_matrix.flatten()[i]))}', ha='center', va='center', color='black', fontsize=5)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid with MAE")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_mae.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

    plt.figure(figsize=(12, 12))
    scatter = plt.scatter(longitudes, latitudes, c=mae_matrix.flatten(), cmap=trunc_cmap, marker='o')
    plt.colorbar(scatter, label='MAE')
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid with MAE")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_mae_clean.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

    # 繪製 MAPE 網格圖
    plt.figure(figsize=(12, 12))
    scatter = plt.scatter(longitudes, latitudes, c=mape_matrix.flatten(), cmap=trunc_cmap, marker='o')
    plt.colorbar(scatter, label='MAPE (%)')
    for i, (lon, lat) in enumerate(zip(longitudes, latitudes)):
        plt.text(lon, lat, f'{int(round(mape_matrix.flatten()[i]))}', ha='center', va='center', color='black', fontsize=7)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid with MAPE")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_mape.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

    plt.figure(figsize=(12, 12))
    scatter = plt.scatter(longitudes, latitudes, c=mape_matrix.flatten(), cmap=trunc_cmap, marker='o')
    plt.colorbar(scatter, label='MAPE (%)')
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid with MAPE")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_mape_clean.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

    # 繪製 SMAPE 網格圖（標示整數）
    if smape_matrix is not None:
        plt.figure(figsize=(12, 12))
        scatter = plt.scatter(longitudes, latitudes, c=smape_matrix.flatten(), cmap=trunc_cmap, marker='o')
        plt.colorbar(scatter, label='SMAPE (%)')
        for i, (lon, lat) in enumerate(zip(longitudes, latitudes)):
            plt.text(lon, lat, f'{int(round(smape_matrix.flatten()[i]))}', ha='center', va='center', color='black', fontsize=7)
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title("Grid with SMAPE")
        plt.grid(True)
        plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_smape.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
        plt.close()

        plt.figure(figsize=(12, 12))
        scatter = plt.scatter(longitudes, latitudes, c=smape_matrix.flatten(), cmap=trunc_cmap, marker='o')
        plt.colorbar(scatter, label='SMAPE (%)')
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title("Grid with SMAPE")
        plt.grid(True)
        plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_smape_clean.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
        plt.close()

    # 更新表格，新增 SMAPE
    table_data = {
        'Grid Index': [f'[{i},{j}]' for i in range(H) for j in range(W)],
        'Longitude': longitudes,
        'Latitude': latitudes,
        'MSE': mse_matrix.flatten(),
        'MAE': mae_matrix.flatten(),
        'MAPE (%)': mape_matrix.flatten()
    }
    if smape_matrix is not None:
        table_data['SMAPE (%)'] = smape_matrix.flatten()
    
    df = pd.DataFrame(table_data)
    df.to_csv(os.path.join(save_dir, 'mse_mae_mape_smape_per_coordinate.csv'), index=False)
    df.to_excel(os.path.join(save_dir, 'mse_mae_mape_smape_per_coordinate.xlsx'), index=False)

def compute_rgb_mean_std(dataset, sample_count=100):
    """
    遍歷部分資料集（例如目標網格轉換後的 RGB 熱力圖），
    計算所有圖像每個通道的平均值與標準差。
    """
    all_pixels = []
    total_samples = min(len(dataset), sample_count)
    
    for idx in range(total_samples):
        # 這裡以目標數據為例，shape (1, 1, H, W)
        _, target, _ = dataset[idx]
        grid = target.squeeze().cpu().numpy()  # (H, W)
        vmin, vmax = grid.min(), grid.max()
        norm_grid = (grid - vmin) / (vmax - vmin + 1e-8)
        # 使用 viridis colormap，得到 (H, W, 4) RGBA，再取前三個通道
        rgb = (plt.cm.viridis(norm_grid)[..., :3] * 255).astype(np.uint8)
        rgb_normalized = rgb.astype(np.float32) / 255.0
        all_pixels.append(rgb_normalized.reshape(-1, 3))
    
    all_pixels = np.concatenate(all_pixels, axis=0)
    mean = np.mean(all_pixels, axis=0)
    std = np.std(all_pixels, axis=0)
    return mean.tolist(), std.tolist()

In [5]:
def train_ddpm(diffusion: DDPM3D, train_loader: DataLoader, val_loader: DataLoader, 
               model_name: str,  # 新增參數
               epochs: int = 20, lr: float = 1e-4, device: str = 'cuda', 
               patience: int = 3, weight_decay: float = 1e-6, 
               save_dir: str = r"C:\\thesis\\code\\result_ddpm_hierarchical",
               checkpoint_interval: int = 5) -> DDPM3D:
    """
    訓練 DDPM 模型，並進行驗證與早停檢查，支援動態模型命名。
    Args:
        diffusion: DDPM 模型實例
        train_loader, val_loader: 訓練與驗證的 DataLoader
        model_name: 模型名稱（例如 'holiday', 'weekday'），用於存檔命名
        epochs: 最大訓練輪數
        lr: 初始學習率
        device: 訓練設備
        patience: 早停耐心次數
        weight_decay: 優化器的權重衰減
        save_dir: 模型與結果的存檔目錄
        checkpoint_interval: 檢查點保存間隔
    Returns:
        訓練後的 diffusion 模型
    """
    for cond, target, extra_data in train_loader:
        print(f"cond shape: {cond.shape}, target shape: {target.shape}, extra_data shape: {extra_data.shape}")
        break
    optimizer = optim.AdamW(diffusion.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)
    diffusion.to(device)
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses = [], []
    lr_history = []

    os.makedirs(save_dir, exist_ok=True)
    checkpoint_path = os.path.join(save_dir, f'{model_name}_checkpoint.pth')

    start_epoch = 0
    for epoch in range(start_epoch, epochs):
        diffusion.train()
        total_train_loss = 0
        for cond, target, extra_data in train_loader:
            cond, target, extra_data = cond.to(device), target.to(device), extra_data.to(device)
            optimizer.zero_grad()
            t = torch.randint(0, diffusion.timesteps, (target.shape[0],), device=device)
            loss = diffusion.p_losses(cond, target, t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(diffusion.parameters(), max_norm=1.0)
            optimizer.step()
            total_train_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        diffusion.eval()
        total_val_loss = 0
        with torch.no_grad():
            for cond, target, extra_data in val_loader:
                cond, target, extra_data = cond.to(device), target.to(device), extra_data.to(device)
                t = torch.randint(0, diffusion.timesteps, (target.shape[0],), device=device)
                loss = diffusion.p_losses(cond, target, t)
                total_val_loss += loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        scheduler.step(avg_val_loss)
        current_lr = scheduler.get_last_lr()[0]
        lr_history.append(current_lr)

        logging.info(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Learning Rate: {current_lr:.8f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save({
                'model_state_dict': diffusion.state_dict(),
                'learning_rate': current_lr,
            }, os.path.join(save_dir, f'{model_name}_best_model.pth'))
            logging.info(f"保存最佳模型，驗證損失: {best_val_loss:.4f}, 學習率: {current_lr:.8f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                logging.info("Early stopping triggered.")
                break

        if (epoch + 1) % checkpoint_interval == 0:
            torch.save({
                'model_state_dict': diffusion.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'epoch': epoch + 1,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'learning_rate': current_lr,
            }, checkpoint_path)
            logging.info(f"在 epoch {epoch + 1} 保存檢查點，學習率: {current_lr:.8f}")

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss')
    plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, f'{model_name}_loss_curve.png'), dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(lr_history) + 1), lr_history, label='Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.title('Learning Rate Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, f'{model_name}_lr_curve.png'), dpi=300, bbox_inches='tight')
    plt.close()

    return diffusion


In [ ]:
@torch.no_grad() # 通常評估指標計算不需要梯度
def calculate_metrics(generated, target):
    """
    計算單一樣本或批次樣本的評估指標。

    Args:
        generated (torch.Tensor): 生成的預測張量。
        target (torch.Tensor): 真實的目標張量。

    Returns:
        dict: 包含 mse, mae, mape, smape 的字典。
    """
    # 添加 epsilon 防止除以零
    epsilon = 1e-10
    mse = F.mse_loss(generated, target).item()
    mae = F.l1_loss(generated, target).item()
    # MAPE: 平均絕對百分比誤差
    mape = torch.mean(torch.abs((target - generated) / (target + epsilon))) * 100
    # SMAPE: 對稱平均絕對百分比誤差 (這裡使用分母為 |target|+|generated| 的常見定義 * 100)
    smape = torch.mean(torch.abs(generated - target) / (torch.abs(target) + torch.abs(generated) + epsilon)) * 100

    return {'mse': mse, 'mae': mae, 'mape': mape.item(), 'smape': smape.item()}

@torch.no_grad()
def evaluate_hierarchical_model(diffusion, error_diffusion, dataset, # dataset 通常是 test_dataset
                                condition_column, condition_value, operator_str='==',
                                device='cuda', max_samples=150,
                                save_dir=r"C:\\thesis\\code\\result_ddpm_hierarchical"):
    """
    評估分層模型（基礎模型+誤差模型）在特定條件下的性能。

    Args:
        diffusion: 訓練好的基礎 DDPM 模型。
        error_diffusion: 訓練好的誤差 DDPM 模型。
        dataset: 用於評估的數據集（通常是測試集，應為 Subset 或完整 Dataset）。
        condition_column (str): 用於篩選數據的欄位名稱。
        condition_value: 篩選條件的值。
        operator_str (str): 比較運算符 ('==', '>', '<', '!=', '>=', '<=').
        device (str): 計算設備。
        max_samples (int): 最大評估樣本數。
        save_dir (str): 結果保存目錄。

    Returns:
        tuple[dict, dict]: 包含基礎模型和分層模型評估指標的字典元組。
                           (metrics_base, metrics_hierarchical)
    """
    diffusion.eval()
    error_diffusion.eval()
    from scipy.linalg import sqrtm # 嘗試在此處導入以解決範圍問題

    # --- 獲取數據集基本屬性 ---
    base_dataset = dataset
    while isinstance(base_dataset, Subset):
        base_dataset = base_dataset.dataset
    H, W = base_dataset.H, base_dataset.W
    pred_length = base_dataset.prediction_length # 應該是 1
    try:
        mean_val = base_dataset.mean_val.to(device)
        std_val = base_dataset.std_val.to(device)
    except AttributeError:
        raise AttributeError("基礎數據集 'base_dataset' 缺少 'mean_val' 或 'std_val' 屬性，請確保數據集已正確初始化。")

    # --- 篩選符合條件的樣本索引 ---
    logging.info(f"正在從提供的數據集 (長度 {len(dataset)}) 中篩選條件: {condition_column} {operator_str} {condition_value}")
    filtered_df = filter_by_condition(base_dataset.df, condition_column, condition_value, operator_str)
    condition_indices_in_base = filtered_df.index.tolist()
    # 確保索引在 base_dataset 範圍內
    condition_indices_in_base = [i for i in condition_indices_in_base if i < len(base_dataset)]
    logging.info(f"在完整數據集中找到 {len(condition_indices_in_base)} 個符合條件的索引。")

    # 將 base dataset 的索引轉換為輸入 dataset (通常是 test_dataset) 中的相對索引
    if isinstance(dataset, Subset):
        subset_indices = dataset.indices # 這是 test_dataset 在 base_dataset 中的索引列表
        # 找出 condition_indices_in_base 中也存在於 test_dataset (subset_indices) 的那些索引
        valid_original_indices = [idx for idx in condition_indices_in_base if idx in subset_indices]
        if not valid_original_indices:
             raise ValueError(f"在提供的數據集子集 (test_dataset) 中找不到符合條件 {condition_column}{operator_str}{condition_value} 的樣本")
        # 將這些有效原始索引轉換為它們在 test_dataset 中的相對索引
        relative_indices = [subset_indices.index(idx) for idx in valid_original_indices]
        logging.info(f"在輸入數據集子集 (長度 {len(dataset)}) 中找到 {len(relative_indices)} 個符合條件的相對索引。")
    else: # 如果輸入的 dataset 不是 Subset，直接使用 base 的索引
        relative_indices = condition_indices_in_base
        logging.info(f"輸入數據集非 Subset，使用 {len(relative_indices)} 個符合條件的索引。")


    # 創建只包含符合條件樣本的子集 (來自輸入的 dataset)
    condition_dataset = Subset(dataset, relative_indices)
    logging.info(f"創建了只包含條件樣本的數據集，長度: {len(condition_dataset)}")
    # --- 篩選邏輯結束 ---

    # --- 確定實際評估樣本數 N ---
    N = min(len(condition_dataset), max_samples)
    if N == 0:
        raise ValueError(f"沒有符合條件 '{condition_column} {operator_str} {condition_value}' 的樣本可供評估 (在提供的 dataset 子集中)")
    logging.info(f"將評估 {N} 個樣本 (從 {len(condition_dataset)} 個可用條件樣本中抽取)。")

    # --- **修正抽樣邏輯** ---
    # 從 *符合條件* 的數據集 condition_dataset 中隨機抽樣 N 個 *相對* 索引
    sample_indices_in_condition_dataset = random.sample(range(len(condition_dataset)), N)
    # --- **修正完成** ---

    # --- 初始化用於存儲結果的張量 ---
    generated_base_batch = torch.zeros(N, 1, pred_length, H, W, device=device)
    generated_hierarchical_batch = torch.zeros(N, 1, pred_length, H, W, device=device)
    target_batch = torch.zeros(N, 1, pred_length, H, W, device=device)

    # --- 迭代處理抽樣的條件樣本 ---
    for i, idx_in_condition_dataset in tqdm(enumerate(sample_indices_in_condition_dataset), total=N, desc="Processing filtered samples"):
        # 從 condition_dataset 中獲取數據 (cond 包含真實目標)
        cond, target, extra_data = condition_dataset[idx_in_condition_dataset]
        cond = cond.to(device)                     # (1, 1, 9, 21, 21)
        target = target.to(device).unsqueeze(2)    # (1, 1, 1, 21, 21)
        # extra_data = extra_data.to(device)       # (1, 8, num_extra_features) - 暫時未使用

        # 確保 cond 是 5D (雖然 collate_fn 後應該是)
        if cond.dim() == 4:
            cond = cond.unsqueeze(1)

        # --- 模型預測 ---
        # 基礎模型預測 (正規化空間)
        base_pred_norm = diffusion.p_sample_loop(target.shape, cond) # (1, 1, 1, 21, 21)
        # 誤差模型預測 (正規化空間中的誤差) - 注意：傳遞的是原始 cond
        error_pred_norm = error_diffusion.p_sample_loop(target.shape, cond) # (1, 1, 1, 21, 21)
        # 分層模型預測 (正規化空間)
        hierarchical_pred_norm = base_pred_norm - error_pred_norm

        # --- 反正規化 ---
        # 分層預測反正規化
        hierarchical_pred = hierarchical_pred_norm * std_val + mean_val
        # 真實目標反正規化
        target_original = target * std_val + mean_val
        # 基礎預測反正規化
        base_pred_original = base_pred_norm * std_val + mean_val

        # --- 儲存結果 ---
        generated_base_batch[i] = base_pred_original
        generated_hierarchical_batch[i] = hierarchical_pred
        target_batch[i] = target_original

    # --- 計算誤差模型預測值並儲存數據 (用於調試分析) ---
    # 注意：這是在原始數據空間計算的誤差模型輸出
    error_pred_batch_original_scale = generated_hierarchical_batch - generated_base_batch

    # 將張量轉換為 NumPy 陣列
    generated_base_np = generated_base_batch.cpu().numpy()
    error_pred_np_original_scale = error_pred_batch_original_scale.cpu().numpy()
    target_np = target_batch.cpu().numpy()

    # 重塑為 (N, pred_length * H * W)
    num_elements = pred_length * H * W
    generated_base_flat = generated_base_np.reshape(N, num_elements)
    error_pred_flat_original_scale = error_pred_np_original_scale.reshape(N, num_elements)
    target_flat = target_np.reshape(N, num_elements)

    # 生成欄位名稱
    columns = []
    for t in range(pred_length): # pred_length 應為 1
        for h in range(H):
            for w in range(W):
                columns.append(f'base_pred_t{t}_h{h}_w{w}')
                columns.append(f'error_pred_original_scale_t{t}_h{h}_w{w}') # 標明是原始尺度
                columns.append(f'target_t{t}_h{h}_w{w}')

    # 創建 DataFrame
    data_to_save = []
    for i in range(N):
        row = []
        # 因為 pred_length = 1, t 實際上只會是 0
        flat_idx_offset = 0
        for t in range(pred_length):
            for h in range(H):
                for w in range(W):
                    idx = flat_idx_offset + h * W + w
                    row.append(generated_base_flat[i, idx])
                    row.append(error_pred_flat_original_scale[i, idx])
                    row.append(target_flat[i, idx])
            flat_idx_offset += H * W # 更新偏移量 (雖然只有一輪)
        data_to_save.append(row)

    df_predictions = pd.DataFrame(data_to_save, columns=columns)

    # 儲存為 CSV
    os.makedirs(save_dir, exist_ok=True)
    csv_path = os.path.join(save_dir, f'evaluation_predictions_{condition_column}_{operator_str}_{condition_value}.csv')
    df_predictions.to_csv(csv_path, index=False)
    logging.info(f"基礎預測、計算出的誤差模型輸出(原始尺度)和真實值已儲存至 {csv_path}")

    # --- 計算基礎模型和階層模型的誤差指標 ---
    metrics_base = {'mse': 0.0, 'mae': 0.0, 'mape': 0.0, 'smape': 0.0}
    metrics_hierarchical = {'mse': 0.0, 'mae': 0.0, 'mape': 0.0, 'smape': 0.0}

    logging.info(f"正在計算 {N} 個樣本的基礎模型和分層模型指標...")
    for i in range(N):
        # 使用反正規化後的值計算指標
        base_sample_metrics = calculate_metrics(generated_base_batch[i], target_batch[i])
        hierarchical_sample_metrics = calculate_metrics(generated_hierarchical_batch[i], target_batch[i])

        for key in metrics_base:
            # 累加每個樣本的指標值
            metrics_base[key] += base_sample_metrics[key]
            metrics_hierarchical[key] += hierarchical_sample_metrics[key]

    # 計算平均指標
    for key in metrics_base:
        metrics_base[key] /= N
        metrics_hierarchical[key] /= N
    logging.info(f"指標計算完成。")

    # --- FID 計算 ---
    logging.info("準備計算 FID...")
    # 1. 計算真實值的特徵
    real_images = []
    # 使用 target_batch (反正規化後的值)
    all_real = target_batch.cpu().numpy() # (N, 1, 1, H, W)
    # 使用百分位數避免極端值影響顏色映射範圍
    global_min = np.percentile(all_real, 1)
    global_max = np.percentile(all_real, 99)
    logging.info(f"用於熱力圖正規化的全局範圍 (1%-99%): min={global_min:.2f}, max={global_max:.2f}")

    for i in range(N):
        for t in range(pred_length): # pred_length = 1
            real_arr = target_batch[i, 0, t].cpu().numpy()
            # 檢查 NaN/Inf
            if not np.all(np.isfinite(real_arr)):
                logging.warning(f"樣本 {i} 的真實數據包含 NaN 或 Inf 值，已替換為 0")
                real_arr = np.nan_to_num(real_arr, nan=0.0, posinf=global_max, neginf=global_min) # 用邊界值替換
            # 正規化到 [0, 1] 用於顏色映射，並進行 clip
            real_norm = np.clip((real_arr - global_min) / (global_max - global_min + 1e-8), 0, 1)
            real_rgb = (plt.cm.viridis(real_norm)[..., :3] * 255).astype(np.uint8)
            real_images.append(Image.fromarray(real_rgb))

    # 2. 計算生成值 (分層模型) 的特徵
    pred_images_hierarchical = []
    for i in range(N):
        for t in range(pred_length): # pred_length = 1
            gen_arr = generated_hierarchical_batch[i, 0, t].cpu().numpy()
            if not np.all(np.isfinite(gen_arr)):
                logging.warning(f"樣本 {i} 的分層模型生成數據包含 NaN 或 Inf 值，已替換為 0")
                gen_arr = np.nan_to_num(gen_arr, nan=0.0, posinf=global_max, neginf=global_min)
            gen_norm = np.clip((gen_arr - global_min) / (global_max - global_min + 1e-8), 0, 1)
            gen_rgb = (plt.cm.viridis(gen_norm)[..., :3] * 255).astype(np.uint8)
            pred_images_hierarchical.append(Image.fromarray(gen_rgb))

    # 3. 計算 Inception 特徵提取所需的均值和標準差 (基於 RGB 熱力圖)
    # 注意：這裡的 dataset 是 evaluate_hierarchical_model 的輸入，即 test_dataset
    # 如果 condition_dataset 很小，可能用 condition_dataset 計算更相關？
    # 或者用更大的數據集（如 val_dataset）預計算一次？維持原狀，用輸入 dataset 計算。
    try:
        # 嘗試從基礎數據集獲取預計算的 mean/std (如果已添加)
        inception_mean = base_dataset.inception_mean
        inception_std = base_dataset.inception_std
        logging.info("使用數據集預計算的 Inception mean/std")
    except AttributeError:
        logging.info("計算 Inception mean/std...")
        # 注意：這裡的 dataset 是原始的 test_dataset，不是 condition_dataset
        inception_mean, inception_std = compute_rgb_mean_std(dataset, sample_count=min(100, len(dataset)))
        # (可選) 將計算結果存回 base_dataset 以便後續使用
        # base_dataset.inception_mean = inception_mean
        # base_dataset.inception_std = inception_std
        logging.info(f"計算完成: mean={inception_mean}, std={inception_std}")


    # 4. 初始化 Inception 模型和轉換
    try:
        inception_model = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1) # aux_logits=False 可能更常用
        inception_model.fc = torch.nn.Identity() # 移除最後分類層
        # inception_model.AuxLogits = None # 如果 aux_logits=False 則不需要這行
        inception_model.to(device)
        inception_model.eval()

        inception_transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize(mean=inception_mean, std=inception_std)
        ])
    except Exception as e:
        logging.error(f"初始化 Inception 模型失敗: {e}")
        metrics_hierarchical['fid'] = float('nan') # 無法計算 FID
        # 如果無法計算 FID，後續的 evaluate_model 也無法接收 precomputed_real_features
        precomputed_real_features = None
    else:
        # 5. 提取特徵
        try:
            real_tensors = torch.stack([inception_transform(img).to(device) for img in real_images])
            pred_tensors_hierarchical = torch.stack([inception_transform(img).to(device) for img in pred_images_hierarchical])

            with torch.no_grad():
                real_features = inception_model(real_tensors).cpu().numpy()
                pred_features_hierarchical = inception_model(pred_tensors_hierarchical).cpu().numpy()

            logging.info(f"真實特徵形狀: {real_features.shape}, 分層模型生成特徵形狀: {pred_features_hierarchical.shape}")

            # 6. 計算分層模型的 FID
            fid_hierarchical = compute_fid(pred_features_hierarchical, real_features) # 使用輔助函數
            logging.info(f"分層模型 FID 計算結果: {fid_hierarchical}")
            metrics_hierarchical['fid'] = fid_hierarchical if not np.isnan(fid_hierarchical) else float('inf')
            # 將計算好的真實特徵傳遞給 evaluate_model
            precomputed_real_features = real_features

        except Exception as e:
             logging.error(f"提取 Inception 特徵或計算 FID 時出錯: {e}")
             metrics_hierarchical['fid'] = float('nan')
             precomputed_real_features = None # 出錯則無法傳遞

    # --- 調用 evaluate_model 分別評估兩個模型 ---
    # 注意: evaluate_model 現在需要接收 num_eval_samples 參數
    # 我們傳遞 N，即實際處理的樣本數
    # 同時傳遞 condition_dataset，讓 evaluate_model 也在過濾後的數據上操作 (雖然預測值已給定)
    # 指定不同的 save_dir 子目錄以區分結果

    base_model_save_dir = os.path.join(save_dir, f"base_model_{condition_column}_{operator_str}_{condition_value}")
    hierarchical_model_save_dir = os.path.join(save_dir, f"hierarchical_model_{condition_column}_{operator_str}_{condition_value}")

    logging.info(f"調用 evaluate_model 評估基礎模型 (在 {N} 個過濾樣本上)...")
    metrics_base_full = evaluate_model(
        diffusion=diffusion,
        dataset=condition_dataset, # 傳遞過濾後的數據集
        device=device,
        num_eval_samples=N, # 傳遞實際樣本數 N
        save_dir=base_model_save_dir,
        sample_idx=0, # 或者選擇一個有效的索引
        error_diffusion=None, # 基礎模型沒有誤差模型
        precomputed_predictions=generated_base_batch,
        precomputed_metrics=metrics_base, # 傳遞預計算的指標
        precomputed_real_features=precomputed_real_features # 傳遞預計算的真實特徵
    )
    logging.info(f"基礎模型評估完成。")

    logging.info(f"調用 evaluate_model 評估分層模型 (在 {N} 個過濾樣本上)...")
    metrics_hierarchical_full = evaluate_model(
        diffusion=diffusion, # 基礎模型仍需傳遞
        dataset=condition_dataset, # 傳遞過濾後的數據集
        device=device,
        num_eval_samples=N, # 傳遞實際樣本數 N
        save_dir=hierarchical_model_save_dir,
        sample_idx=0, # 或者選擇一個有效的索引
        error_diffusion=error_diffusion, # 傳遞誤差模型
        precomputed_predictions=generated_hierarchical_batch,
        precomputed_metrics=metrics_hierarchical, # 傳遞預計算的指標 (包含 FID)
        precomputed_real_features=precomputed_real_features # 傳遞預計算的真實特徵
    )
    logging.info(f"分層模型評估完成。")


    # --- 計算兩者指標差異並保存 ---
    differences = {}
    logging.info("計算指標差異 (基礎模型 - 分層模型)...")
    for key in ['mse', 'mae', 'mape', 'smape', 'fid']:
        # 確保兩個字典中都有該鍵，並且值不是 NaN
        base_val = metrics_base_full.get(key, float('nan'))
        hierarchical_val = metrics_hierarchical_full.get(key, float('nan'))
        if not np.isnan(base_val) and not np.isnan(hierarchical_val):
            differences[key.upper()] = base_val - hierarchical_val
        else:
            differences[key.upper()] = float('nan') # 如果任一為 NaN，差異也為 NaN
        logging.info(f"指標 {key.upper()}: Base={base_val:.4f}, Hierarchical={hierarchical_val:.4f}, Difference={differences[key.upper()]:.4f}")

    # 建立匯總結果的目錄
    summary_save_dir = os.path.join(save_dir, "evaluation_summary")
    os.makedirs(summary_save_dir, exist_ok=True)
    diff_filename = f'metrics_differences_{condition_column}_{operator_str}_{condition_value}.json'
    diff_path = os.path.join(summary_save_dir, diff_filename)
    try:
        with open(diff_path, 'w') as f:
            json.dump(differences, f, indent=4)
        logging.info(f"指標差異已保存至 {diff_path}")
    except Exception as e:
        logging.error(f"保存指標差異失敗: {e}")

    # --- 返回最終評估結果 ---
    return metrics_base_full, metrics_hierarchical_full

# --- 輔助函數 compute_fid (確保 sqrtm 可用) ---
def compute_fid(pred_features, real_features):
    """
    計算 FID（Fréchet Inception Distance）。

    參數:
        pred_features: 生成圖像的特徵 (numpy array)
        real_features: 真實圖像的特徵 (numpy array)

    返回:
        FID 值，若計算失敗則返回 np.nan
    """
    try:
        from scipy.linalg import sqrtm # 在函數內部導入，嘗試解決 scope 問題
    except ImportError:
        logging.error("無法導入 scipy.linalg.sqrtm，請確保已安裝 scipy。")
        return np.nan

    try:
        # 檢查輸入維度
        if pred_features.ndim != 2 or real_features.ndim != 2:
             raise ValueError(f"特徵必須是 2D 陣列，但得到維度 {pred_features.ndim} 和 {real_features.ndim}")
        if pred_features.shape[0] < 2 or real_features.shape[0] < 2:
             logging.warning(f"計算 FID 的樣本數過少 ({pred_features.shape[0]}, {real_features.shape[0]})，結果可能不可靠。")
             # return np.nan # 或者繼續計算但發出警告

        # 計算均值和協方差
        mu1, sigma1 = np.mean(pred_features, axis=0), np.cov(pred_features, rowvar=False)
        mu2, sigma2 = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)

        # 檢查數據有效性
        if not (np.all(np.isfinite(mu1)) and np.all(np.isfinite(mu2)) and
                np.all(np.isfinite(sigma1)) and np.all(np.isfinite(sigma2))):
            logging.error("特徵數據包含 NaN 或 Inf 值，無法計算 FID。")
            # 可以嘗試找出問題來源：
            # logging.error(f"Pred features finite: {np.all(np.isfinite(pred_features))}")
            # logging.error(f"Real features finite: {np.all(np.isfinite(real_features))}")
            return np.nan

        # 添加正則化項以提高數值穩定性
        eps = 1e-6 # 稍微調整 epsilon
        sigma1 += np.eye(sigma1.shape[0]) * eps
        sigma2 += np.eye(sigma2.shape[0]) * eps

        # 計算均值差的平方
        ssdiff = np.sum((mu1 - mu2) ** 2.0)

        # 計算協方差矩陣乘積的平方根
        covmean, _ = sqrtm(sigma1.dot(sigma2), disp=False) # 使用 disp=False 抑制警告

        # 處理複數結果 (雖然理論上不應出現，但以防萬一)
        if np.iscomplexobj(covmean):
            # logging.warning("FID 計算中 covmean 包含虛部，將取實部。")
            covmean = covmean.real

        # 計算 FID
        fid = ssdiff + np.trace(sigma1 + sigma2 - 2.0 * covmean)

        # FID 應該非負
        if fid < 0:
             logging.warning(f"計算出的 FID 為負值 ({fid:.4f})，可能由數值不穩定引起，將其設為 0。")
             fid = 0.0

        return fid
    except ValueError as ve:
        # 捕捉如協方差矩陣不可逆等問題
        logging.error(f"計算 FID 時發生 ValueError: {ve}")
        return np.nan
    except Exception as e:
        logging.error(f"計算 FID 時發生未預期錯誤: {e}")
        import traceback
        traceback.print_exc() # 打印詳細錯誤追蹤
        return np.nan


@torch.no_grad()
def evaluate_model(diffusion: DDPM3D, dataset: Dataset, device: str = 'cuda',
                   num_eval_samples: int = 100, # <--- 修改參數名和意義
                   save_dir: str = r"C:\\thesis\\code\\result_ddpm_hierarchical",
                   sample_idx: int = 0, # 用於單樣本視覺化
                   error_diffusion: Optional[DDPM3D] = None,
                   precomputed_predictions: Optional[torch.Tensor] = None,
                   precomputed_metrics: Optional[dict] = None,
                   precomputed_real_features: Optional[np.ndarray] = None) -> dict:
    """
    (更新後) 評估單個模型 (基礎或分層) 的性能。
    現在接收確切的評估樣本數 num_eval_samples。
    """
    diffusion.eval()
    if error_diffusion is not None:
        model_type = "hierarchical_model"
        error_diffusion.eval()
        logging.info(f"正在評估分層模型，樣本數: {num_eval_samples}")
    else:
        model_type = "base_model"
        logging.info(f"正在評估基礎模型，樣本數: {num_eval_samples}")

    # 確保保存目錄存在
    os.makedirs(save_dir, exist_ok=True)

    # 初始化指標字典
    metrics = {'mse': 0.0, 'mae': 0.0, 'mape': 0.0, 'smape': 0.0, 'fid': float('nan')} # 預設 FID 為 NaN
    N = num_eval_samples # 使用傳入的確切樣本數

    # 獲取數據集屬性
    base_dataset = dataset
    while isinstance(base_dataset, Subset):
        base_dataset = base_dataset.dataset
    H, W = base_dataset.H, base_dataset.W
    pred_length = base_dataset.prediction_length # 應為 1
    try:
        mean_val = base_dataset.mean_val.to(device)
        std_val = base_dataset.std_val.to(device)
    except AttributeError:
         raise AttributeError("基礎數據集 'base_dataset' 缺少 'mean_val' 或 'std_val' 屬性。")

    generated_batch = None
    target_batch = None

    # --- 處理預測值和目標值 ---
    if precomputed_predictions is not None:
        if precomputed_predictions.shape[0] != N:
             raise ValueError(f"預計算的預測數量 {precomputed_predictions.shape[0]} 與請求評估數 {N} 不符")
        generated_batch = precomputed_predictions.to(device) # 確保在正確設備上

        # 需要從 dataset 加載對應的 N 個 target
        # 注意：dataset 現在是過濾後的 condition_dataset，長度可能大於 N
        # 我們需要 N 個 target，且它們必須對應 precomputed_predictions
        # 由於 precomputed_predictions 是基於 sample_indices_in_condition_dataset 生成的，
        # 我們需要使用相同的索引從 condition_dataset 提取 target
        target_batch = torch.zeros(N, 1, pred_length, H, W, device=device)
        # 假設 evaluate_hierarchical_model 傳遞了正確的 dataset (condition_dataset)
        # 並且 precomputed_predictions 的順序與 sample_indices_in_condition_dataset 一致
        # 我們需要一種方法獲取這些索引，或者修改 evaluate_hierarchical_model 也傳遞 target_batch
        # *** 為了簡化，我們假設 evaluate_hierarchical_model 已經正確生成並傳遞了 target_batch ***
        # *** 如果沒有，這裡需要修改 ***
        # 假設 target_batch 也是預計算並傳入的 (例如，通過修改 evaluate_hierarchical_model 返回它)
        # 或者，如果 precomputed_real_features 存在，我們可以認為 target 已處理

        # 暫時的解決方案：重新從 dataset (condition_dataset) 中按順序提取前 N 個 target
        logging.info("evaluate_model 正在從 dataset 中提取前 N 個 target，假設與預計算預測對應。")
        temp_targets = []
        if N > len(dataset):
             logging.error(f"請求樣本數 {N} 大於數據集長度 {len(dataset)}")
             N = len(dataset) # 修正 N
             generated_batch = generated_batch[:N] # 裁剪預測

        for i in range(N):
             _, target, _ = dataset[i] # 從 condition_dataset 取前 N 個
             temp_targets.append(target.unsqueeze(2)) # (1, 1, 1, H, W)
        target_batch = torch.cat(temp_targets, dim=0).to(device)
        # 反正規化 (如果在 precomputed 中尚未完成)
        if torch.max(target_batch) < 10: # 簡易判斷是否為正規化值
             target_batch = target_batch * std_val + mean_val


    else:
        # 如果沒有預計算預測，則需要自行生成
        logging.info(f"沒有預計算預測，將從 dataset (長度 {len(dataset)}) 中抽樣 {N} 個樣本生成預測...")
        if N > len(dataset):
             logging.warning(f"請求評估樣本數 {N} 大於數據集大小 {len(dataset)}，將使用 {len(dataset)} 個樣本")
             N = len(dataset)
        sample_indices_in_dataset = random.sample(range(len(dataset)), N)

        generated_batch = torch.zeros(N, 1, pred_length, H, W, device=device)
        target_batch = torch.zeros(N, 1, pred_length, H, W, device=device)

        for i, idx in tqdm(enumerate(sample_indices_in_dataset), total=N, desc=f"Eval {model_type} samples"):
            cond, target, _ = dataset[idx]
            cond, target = cond.to(device), target.to(device)
            target = target.unsqueeze(2)

            if cond.dim() == 4: cond = cond.unsqueeze(1)

            # 模型預測 (正規化空間)
            x_recon_norm = diffusion.p_sample_loop(target.shape, cond)
            if error_diffusion is not None:
                error_pred_norm = error_diffusion.p_sample_loop(target.shape, cond)
                x_recon_norm = x_recon_norm + error_pred_norm

            # 反正規化
            x_recon_original = x_recon_norm * std_val + mean_val
            target_original = target * std_val + mean_val

            generated_batch[i] = x_recon_original
            target_batch[i] = target_original

    # --- 計算基礎指標 (MSE, MAE, MAPE, SMAPE) ---
    if precomputed_metrics:
        # 如果傳入了預計算指標，直接使用 (除了 FID)
        metrics.update({k: v for k, v in precomputed_metrics.items() if k != 'fid'})
        logging.info("使用了預計算的 MSE, MAE, MAPE, SMAPE 指標。")
    else:
        # 否則，重新計算
        logging.info(f"正在計算 {N} 個樣本的 MSE, MAE, MAPE, SMAPE 指標...")
        for i in range(N):
            sample_metrics = calculate_metrics(generated_batch[i], target_batch[i])
            metrics['mse'] += sample_metrics['mse']
            metrics['mae'] += sample_metrics['mae']
            metrics['mape'] += sample_metrics['mape']
            metrics['smape'] += sample_metrics['smape']
        metrics['mse'] /= N
        metrics['mae'] /= N
        metrics['mape'] /= N
        metrics['smape'] /= N
        logging.info("指標計算完成。")


    # --- FID 計算 ---
    num_images_for_fid = N * pred_length
    logging.info(f"準備計算 FID，需要 {num_images_for_fid} 張圖像...")

    inception_model = None
    inception_transform = None
    inception_mean = None
    inception_std = None
    real_features = None # 初始化 real_features
    pred_features = None # 初始化 pred_features
    metrics['fid'] = float('nan') # 預設 FID 為 NaN

    # --- 1. 確保 target_batch 存在 ---
    if target_batch is None:
        logging.error("target_batch 為空，無法進行 FID 計算。")
    else:
        all_real = target_batch.cpu().numpy()
        global_min = np.percentile(all_real, 1)
        global_max = np.percentile(all_real, 99)
        logging.info(f"用於熱力圖正規化的全局範圍 (1%-99%): min={global_min:.2f}, max={global_max:.2f}")

        # --- 2. 獲取或計算 Inception mean/std ---
        try:
            inception_mean = base_dataset.inception_mean
            inception_std = base_dataset.inception_std
            logging.info("使用數據集預計算的 Inception mean/std")
        except AttributeError:
            logging.info("計算 Inception mean/std...")
            try:
                # dataset 參數現在是 condition_dataset
                inception_mean, inception_std = compute_rgb_mean_std(dataset, sample_count=min(100, len(dataset)))
                # (可選) 將計算結果存回 base_dataset 以便後續使用
                # base_dataset.inception_mean = inception_mean
                # base_dataset.inception_std = inception_std
                logging.info(f"計算完成: mean={inception_mean}, std={inception_std}")
            except Exception as e:
                logging.error(f"計算 Inception mean/std 時出錯: {e}")
                # 如果無法計算 mean/std，後續也無法進行
                inception_mean, inception_std = None, None

        # --- 3. 初始化 Inception 模型和轉換 (只需一次) ---
        if inception_mean is not None and inception_std is not None:
            try:
                inception_model = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1) # 移除 aux_logits=False
                inception_model.fc = torch.nn.Identity()
                inception_model.to(device)
                inception_model.eval()

                inception_transform = transforms.Compose([
                    transforms.Resize((299, 299)),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=inception_mean, std=inception_std)
                ])
                logging.info("Inception 模型和轉換初始化成功。")
            except Exception as e:
                logging.error(f"初始化 Inception 模型或轉換時出錯: {e}")
                inception_model = None # 標記初始化失敗
                inception_transform = None
        else:
             logging.error("缺少 Inception mean/std，無法初始化模型和轉換。")


    # --- 4. 計算真實特徵 (如果需要且模型/轉換可用) ---
    if inception_model is not None and inception_transform is not None:
        if precomputed_real_features is not None:
            if precomputed_real_features.shape[0] != num_images_for_fid:
                 logging.warning(f"預計算的真實特徵數量 {precomputed_real_features.shape[0]} 與預期 {num_images_for_fid} 不符，將重新計算。")
                 real_features = None
            else:
                 real_features = precomputed_real_features
                 logging.info("使用預計算的真實圖像 Inception 特徵。")
        else:
            real_features = None

        if real_features is None:
            logging.info("計算真實圖像的 Inception 特徵...")
            real_images = []
            for i in range(N):
                for t in range(pred_length):
                    real_arr = target_batch[i, 0, t].cpu().numpy()
                    if not np.all(np.isfinite(real_arr)):
                        real_arr = np.nan_to_num(real_arr, nan=0.0, posinf=global_max, neginf=global_min)
                    real_norm = np.clip((real_arr - global_min) / (global_max - global_min + 1e-8), 0, 1)
                    real_rgb = (plt.cm.viridis(real_norm)[..., :3] * 255).astype(np.uint8)
                    real_images.append(Image.fromarray(real_rgb))

            if len(real_images) < 2:
                logging.error("真實圖像數量不足 (<2)，無法計算 FID。")
                real_features = None # 標記計算失敗
            else:
                try:
                    real_tensors = torch.stack([inception_transform(img).to(device) for img in real_images])
                    with torch.no_grad():
                        real_features = inception_model(real_tensors).cpu().numpy()
                    logging.info(f"真實特徵計算完成，形狀: {real_features.shape}")
                except Exception as e:
                    logging.error(f"計算真實特徵時出錯: {e}")
                    real_features = None # 標記計算失敗

    # --- 5. 計算生成特徵 (如果真實特徵計算成功且模型/轉換可用) ---
    if real_features is not None and inception_model is not None and inception_transform is not None:
        logging.info("計算生成圖像的 Inception 特徵...")
        pred_images = []
        for i in range(N):
            for t in range(pred_length):
                gen_arr = generated_batch[i, 0, t].cpu().numpy()
                if not np.all(np.isfinite(gen_arr)):
                    gen_arr = np.nan_to_num(gen_arr, nan=0.0, posinf=global_max, neginf=global_min)
                # 使用前面計算好的 global_min, global_max
                gen_norm = np.clip((gen_arr - global_min) / (global_max - global_min + 1e-8), 0, 1)
                gen_rgb = (plt.cm.viridis(gen_norm)[..., :3] * 255).astype(np.uint8)
                pred_images.append(Image.fromarray(gen_rgb))

        if len(pred_images) < 2:
             logging.error("生成圖像數量不足 (<2)，無法計算 FID。")
             pred_features = None # 標記計算失敗
        else:
            try:
                 pred_tensors = torch.stack([inception_transform(img).to(device) for img in pred_images])
                 with torch.no_grad():
                     pred_features = inception_model(pred_tensors).cpu().numpy()
                 logging.info(f"生成特徵計算完成，形狀: {pred_features.shape}")
            except Exception as e:
                 logging.error(f"計算生成特徵時出錯: {e}")
                 pred_features = None # 標記計算失敗

        # --- 6. 計算 FID (如果 pred_features 也計算成功) ---
        if pred_features is not None:
             fid = compute_fid(pred_features, real_features)
             logging.info(f"FID 計算結果: {fid}")
             metrics['fid'] = fid if not np.isnan(fid) else float('nan') # 如果 compute_fid 內部返回 nan，這裡也設為 nan
        else:
             logging.warning("由於生成特徵未成功計算，跳過 FID 計算。")
             metrics['fid'] = float('nan')

    else:
         # 如果真實特徵未計算成功，或模型/轉換未初始化成功，則跳過 FID
         logging.warning("由於真實特徵未計算或模型/轉換未初始化，跳過生成特徵和 FID 計算。")
         metrics['fid'] = float('nan')


    # --- 儲存網格誤差圖和指標 ---
    # (這部分邏輯與原 evaluate_model 類似，但基於 N 個樣本計算)
    logging.info("計算並保存網格誤差圖和匯總指標...")
    try:
        # 計算每個網格點的平均誤差 (基於 N 個樣本)
        error_matrix_mse = (generated_batch - target_batch) ** 2
        mse_matrix = torch.mean(error_matrix_mse, dim=0).squeeze(0).squeeze(0).cpu().numpy() # 移除 batch 和 time 維度

        error_matrix_mae = torch.abs(generated_batch - target_batch)
        mae_matrix = torch.mean(error_matrix_mae, dim=0).squeeze(0).squeeze(0).cpu().numpy()

        # 計算 MAPE 和 SMAPE 時要特別小心分母為零
        target_batch_cpu = target_batch.cpu() # 轉到 CPU 計算
        generated_batch_cpu = generated_batch.cpu()
        mape_matrix_all = torch.abs((target_batch_cpu - generated_batch_cpu) / (target_batch_cpu + 1e-10)) * 100
        smape_matrix_all = torch.abs(generated_batch_cpu - target_batch_cpu) / \
                           (torch.abs(target_batch_cpu) + torch.abs(generated_batch_cpu) + 1e-10) * 100

        mape_matrix = torch.mean(mape_matrix_all, dim=0).squeeze(0).squeeze(0).numpy()
        smape_matrix = torch.mean(smape_matrix_all, dim=0).squeeze(0).squeeze(0).numpy()

        # 保存網格誤差 CSV
        table_data = {
            'Grid Index': [f'[{i},{j}]' for i in range(H) for j in range(W)],
            'Longitude': [parse_lat_lon(col)[0] for col in base_dataset.sorted_flow_columns],
            'Latitude': [parse_lat_lon(col)[1] for col in base_dataset.sorted_flow_columns],
            'MSE': mse_matrix.flatten(),
            'MAE': mae_matrix.flatten(),
            'MAPE (%)': mape_matrix.flatten(),
            'SMAPE (%)': smape_matrix.flatten(),
        }
        df_grid_errors = pd.DataFrame(table_data)
        grid_csv_path = os.path.join(save_dir, 'mse_mae_mape_smape_per_coordinate.csv')
        df_grid_errors.to_csv(grid_csv_path, index=False)

        # 繪製網格誤差圖
        plot_grid_with_error(base_dataset.sorted_flow_columns, H, W, mse_matrix, mae_matrix, mape_matrix, save_dir, smape_matrix)

        # 繪製平均預測圖
        visualize_predictions(None, generated_batch, target_batch, sample_idx=None, save_dir=save_dir) # sample_idx=None 繪製平均圖

        # 保存單個樣本的預測圖 (如果 sample_idx 有效)
        if 0 <= sample_idx < N:
             visualize_predictions(None, # cond 數據未在此處傳遞，可忽略
                                 generated_batch, target_batch,
                                 sample_idx=sample_idx, save_dir=save_dir)
        else:
             logging.warning(f"提供的 sample_idx ({sample_idx}) 超出範圍 [0, {N-1}]，不繪製單樣本圖。")

    except Exception as e:
        logging.error(f"保存網格誤差或繪圖時出錯: {e}")
        import traceback
        traceback.print_exc()

    # --- 保存匯總評估指標 ---
    metrics_to_save = {
        "mse": metrics['mse'],
        "mae": metrics['mae'],
        "mape": metrics['mape'],
        "smape": metrics['smape'],
        "fid": metrics['fid'], # 保存計算出的 FID (可能是 NaN)
        "sample_size": N,
        "model_type": model_type,
        "timestamp": pd.Timestamp.now(tz='Asia/Taipei').isoformat() # 使用台北時間
    }
    metrics_json_path = os.path.join(save_dir, 'evaluation_metrics.json')
    metrics_txt_path = os.path.join(save_dir, 'evaluation_metrics.txt')
    try:
        with open(metrics_json_path, 'w') as f:
            json.dump(metrics_to_save, f, indent=4)
        with open(metrics_txt_path, 'w') as f:
            f.write(f"Evaluation Metrics ({model_type} - computed on {N} samples):\n")
            for key, value in metrics_to_save.items():
                 if isinstance(value, float):
                     f.write(f"{key.upper()}: {value:.6f}\n")
                 else:
                     f.write(f"{key.upper()}: {value}\n")
        logging.info(f"評估指標已保存至 {save_dir}")
    except Exception as e:
        logging.error(f"保存評估指標失敗: {e}")


    return metrics # 返回包含所有計算指標的字典

In [7]:
# 複合條件
# 載入數據集
dataset = PeopleFlowDatasetCondition(
    csv_path=r"C:\\thesis\\code\\Taipei_CF\\all_merged.csv",
    H=21, W=21, condition_length=8, prediction_length=1,
    normalize=True, debug=False
)
# 加載 BASE MODEL
# 載入基礎模型
device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint_path = r"C:\\thesis\\code\\result_ddpm\\best_model.pth"
checkpoint = torch.load(checkpoint_path, map_location=device)
unet = UNet3D(in_channels=1, base_channels=64, time_emb_dim=128, dropout_rate=0.2)
diffusion = DDPM3D(
    model=unet,
    timesteps=1500,
    beta_start=1e-4,
    beta_end=0.02,
    device=device,
    #extra_columns=dataset.extra_columns  # 傳遞 extra_columns
)
diffusion.load_state_dict(checkpoint['model_state_dict'], strict=False)
diffusion.to(device)
diffusion.eval()


# 定義 ErrorDataset 類別
class ErrorDataset(Dataset):
    # 注意：__init__ 的參數改變了
    def __init__(self, original_conds, target_errors, extra_data, condition_length=8, prediction_length=1):
        # original_conds 形狀: (num_samples, 1, 9, 21, 21) - 包含條件和真實目標
        # target_errors 形狀: (num_samples, 1, 1, 21, 21) - 只有目標時間步的誤差
        # extra_data 形狀: (num_samples, 8, num_extra_features)
        self.original_conds = original_conds
        self.target_errors = target_errors
        self.extra_data = extra_data # 確保這個 extra_data 與 original_conds/target_errors 的樣本是對應的
        self.condition_length = condition_length
        self.prediction_length = prediction_length # 這裡恆為 1

        # (可選) 計算並保存誤差的均值和標準差，用於反正規化（如果需要）
        # self.error_mean = target_errors.mean()
        # self.error_std = target_errors.std() + 1e-5
        # 注意：DDPM 通常在正規化空間操作，可能不需要對誤差本身再正規化

    def __len__(self):
        return len(self.original_conds)

    def __getitem__(self, idx):
        # 模型輸入：原始的條件序列（包含真實目標）
        model_input_cond = self.original_conds[idx] # (1, 9, 21, 21)
        # 模型目標：計算出的目標時間步誤差
        target_error = self.target_errors[idx]      # (1, 1, 21, 21)
        # 額外數據
        extra = self.extra_data[idx]                # (8, num_extra_features)

        # 返回 (模型輸入條件, 目標誤差, 額外數據)
        # 注意：返回結構與 PeopleFlowDatasetCondition 一致，方便 collate_fn 和訓練循環
        return model_input_cond, target_error, extra

def filter_by_condition(df, condition_column, condition_value, operator_str='=='):
    """
    根據指定的條件篩選數據框。
    
    參數:
        df: 輸入數據框 (pandas DataFrame)
        condition_column: 條件欄位名稱 (例如 '氣溫', 'holiday')
        condition_value: 條件目標值
        operator_str: 比較運算符 ('==', '>', '<', '!=', '>=', '<=')
    
    返回:
        篩選後的數據框
    """
    operators = {
        '==': operator.eq,
        '!=': operator.ne,
        '>': operator.gt,
        '<': operator.lt,
        '>=': operator.ge,
        '<=': operator.le
    }
    
    if operator_str not in operators:
        raise ValueError(f"不支援的運算符: {operator_str}")
    
    condition_op = operators[operator_str]
    mask = condition_op(df[condition_column], condition_value)
    return df[mask].copy()

# 定義條件數據生成函數
def generate_condition_errors(dataset, diffusion, device, condition_column, condition_value, operator_str='==', num_samples=1500):
    # 過濾數據集以獲取符合條件的索引
    filtered_df = filter_by_condition(dataset.df, condition_column, condition_value, operator_str)
    condition_indices = filtered_df.index.tolist()

    # 確保索引不超出 dataset 範圍
    valid_indices_in_base = [i for i in condition_indices if i < len(dataset)]

    # 若樣本數超過 num_samples，隨機選取指定數量的索引 (或者依序選取)
    if num_samples is not None and len(valid_indices_in_base) > num_samples:
        # 改為隨機抽樣，避免只取早期數據
        selected_indices = random.sample(valid_indices_in_base, num_samples)
        # 如果需要依序選取（如原碼），用下面這行替換
        # selected_indices = valid_indices_in_base[:num_samples]
    else:
        selected_indices = valid_indices_in_base

    if not selected_indices:
         raise ValueError(f"在數據集中找不到符合條件 {condition_column}{operator_str}{condition_value} 的樣本")

    # 使用選定的索引創建子集
    condition_subset = Subset(dataset, selected_indices)
    # 注意：batch_size=1 確保每次處理一個完整序列
    condition_loader = DataLoader(condition_subset, batch_size=1, shuffle=False, collate_fn=collate_fn)

    all_original_conds = []
    all_target_errors = []
    all_extra_data = [] # 同時收集 extra_data

    for cond, target, extra_data in tqdm(condition_loader, desc=f"Generating {condition_column}{operator_str}{condition_value} simulations"):
        cond, target = cond.to(device), target.to(device) # cond: (1, 1, 9, 21, 21), target: (1, 1, 1, 21, 21)
        extra_data = extra_data.to(device) # (1, 8, num_extra_features)

        with torch.no_grad():
            # 基礎模型預測 (返回的是正規化後的值)
            # target.shape 是 (1, 1, 1, 21, 21)，符合 p_sample_loop 對 shape 的要求
            simulated_target_norm = diffusion.p_sample_loop(target.shape, cond) # (1, 1, 1, 21, 21)

        # 計算目標時間步的誤差 (在正規化空間中)
        # target 是真實目標的正規化值
        error_norm = target - simulated_target_norm # (1, 1, 1, 21, 21)

        all_original_conds.append(cond.cpu()) # 儲存原始條件序列 (含真實目標)
        all_target_errors.append(error_norm.cpu()) # 儲存計算出的目標誤差
        all_extra_data.append(extra_data.cpu()) # 儲存對應的額外數據

    # 合併數據
    original_conds_tensor = torch.cat(all_original_conds, dim=0) # (N, 1, 9, 21, 21)
    target_errors_tensor = torch.cat(all_target_errors, dim=0)   # (N, 1, 1, 21, 21)
    extra_data_tensor = torch.cat(all_extra_data, dim=0)         # (N, 8, num_extra_features)

    print(f"Target Errors shape: {target_errors_tensor.shape}")
    print(f"Target Errors min: {target_errors_tensor.min()}, max: {target_errors_tensor.max()}, mean: {target_errors_tensor.mean()}, std: {target_errors_tensor.std()}")

    return original_conds_tensor.float(), target_errors_tensor.float(), extra_data_tensor.float()

def create_error_dataset(dataset, diffusion, device, condition_length, prediction_length,
                         condition_column, condition_value, operator_str='==', normalize=True, # normalize 參數可能不再需要
                         num_samples=80):
    if condition_column not in dataset.df.columns:
        raise ValueError(f"欄位 {condition_column} 不存在於 dataset.df 中")

    # 調用修改後的函數，獲取原始條件、目標誤差和額外數據
    original_conds, target_errors, extra_data = generate_condition_errors(
        dataset, diffusion, device, condition_column, condition_value,
        operator_str, num_samples=num_samples
    )

    # 使用獲取的數據創建 ErrorDataset
    return ErrorDataset(original_conds, target_errors, extra_data, condition_length, prediction_length)

# (位於 Cell [7] 或包含此函數定義的 Cell)
# 修改後的 save_error_to_csv
def save_error_to_csv(target_errors, save_dir, condition_column, operator_str, condition_value, num_samples):
    """
    (修改版) 將目標時間步的誤差數據儲存為 CSV 檔案。

    參數:
        target_errors: 目標時間步的誤差數據 (PyTorch 張量或 NumPy 陣列，形狀為 (num_samples, 1, 1, H, W))
        save_dir: 儲存檔案的目錄
        condition_column: 條件欄位名稱
        operator_str: 比較運算符
        condition_value: 條件目標值
        num_samples: 樣本數量 (應與 target_errors 的第一維匹配)

    輸出:
        儲存 CSV 檔案，名稱格式為 {num_samples}_{condition_column}_{operator_str英文敘述}_{condition_value}_target_errors.csv
    """
    # 運算符到英文敘述的映射
    operator_map = {
        '==': 'equals', '!=': 'not_equals', '>': 'greater_than',
        '<': 'less_than', '>=': 'greater_than_or_equals', '<=': 'less_than_or_equals'
    }

    if operator_str not in operator_map:
        raise ValueError(f"不支援的運算符: {operator_str}")

    # 將 target_errors 轉換為 NumPy 陣列
    if isinstance(target_errors, torch.Tensor):
        errors_np = target_errors.cpu().numpy()
    else:
        errors_np = target_errors

    # 檢查形狀是否符合預期 (N, 1, 1, H, W)
    if errors_np.ndim != 5 or errors_np.shape[1] != 1 or errors_np.shape[2] != 1:
         raise ValueError(f"輸入 target_errors 的形狀應為 (N, 1, 1, H, W)，但得到 {errors_np.shape}")

    # 獲取形狀參數
    actual_num_samples, _, T, H, W = errors_np.shape # T 應為 1
    if actual_num_samples != num_samples:
        logging.warning(f"提供的 num_samples ({num_samples}) 與實際誤差數據樣本數 ({actual_num_samples}) 不符，將使用實際數量。")
        num_samples = actual_num_samples

    num_elements = H * W  # 每個樣本的元素數量 (因為 T=1)

    # 重塑誤差數據為 (num_samples, num_elements)
    errors_reshaped = errors_np.reshape(num_samples, num_elements)

    # 動態生成列名 (只包含目標時間步 t=0)
    columns = [f'target_error_h{h}_w{w}' for h in range(H) for w in range(W)]

    # 創建 DataFrame
    error_df = pd.DataFrame(errors_reshaped, columns=columns)

    # 確保儲存目錄存在
    os.makedirs(save_dir, exist_ok=True)

    # 使用英文敘述格式化檔案名稱 (添加 _target_errors 區分)
    filename = f"{num_samples}_{condition_column}_{operator_map[operator_str]}_{condition_value}_target_errors.csv"
    csv_path = os.path.join(save_dir, filename)
    error_df.to_csv(csv_path, index=False)
    logging.info(f"目標誤差數據已儲存至 {csv_path}")

def load_error_dataset(csv_path, dataset, condition_length=8, prediction_length=1, normalize=True):
    """
    從 CSV 檔案載入誤差數據，並重建 ErrorDataset。
    
    參數:
        csv_path: CSV 檔案路徑 (例如 '80_holiday_equals_0_生成資料.csv')
        dataset: 原始數據集 (PeopleFlowDatasetCondition)，用於提供 extra_data
        condition_length: 條件序列長度
        prediction_length: 預測序列長度
        normalize: 是否正規化誤差數據
    
    返回:
        ErrorDataset 實例
    """
    # 載入 CSV 檔案
    error_df = pd.read_csv(csv_path)
    
    # 將 DataFrame 轉換為 NumPy 陣列，形狀為 (num_samples, 21*21)
    errors_np = error_df.to_numpy()
    
    # 將誤差數據重塑為 (num_samples, 1, 21, 21)
    num_samples = errors_np.shape[0]
    errors_np = errors_np.reshape(num_samples, 1, 21, 21).astype(np.float32)
    
    # 將 NumPy 陣列轉換為 PyTorch 張量
    errors = torch.from_numpy(errors_np)
    
    # 使用原始數據集的 extra_data
    extra_data = dataset.extra_data
    
    # 創建 ErrorDataset
    return ErrorDataset(errors, extra_data, condition_length, prediction_length, normalize)



訊息: 座標點數量 (495) 多於網格數 (441). 將選擇最靠近地理中心的 441 個座標點進行映射。


In [10]:
num_samples = 1440
condition_column = 'holiday'

In [11]:

# 生成 X 個假日
error_dataset_holiday = create_error_dataset(
    dataset=dataset,
    diffusion=diffusion,
    device='cuda',
    condition_length=8,
    prediction_length=1,
    condition_column=condition_column,
    condition_value=1,
    operator_str='==',
    normalize=True,
    num_samples=num_samples
)

# 生成 100 個氣溫大於 30 度
# error_dataset_high_temp = create_error_dataset(
#     dataset=dataset,
#     diffusion=diffusion,
#     device='cuda',
#     condition_length=8,
#     prediction_length=1,
#     condition_column='氣溫',
#     condition_value=30,
#     operator_str='>',
#     normalize=True,
#     save_dir=r"C:\\thesis\\code\\result_ddpm_hierarchical",
#     num_samples=100
# )

#50 個非假日
# error_dataset_non_holiday = create_error_dataset(
#     dataset=dataset,
#     diffusion=diffusion,
#     device='cuda',
#     condition_length=8,
#     prediction_length=1,
#     condition_column='holiday',
#     condition_value=1,
#     operator_str='!=',
#     normalize=True,
#     save_dir=r"C:\\thesis\\code\\result_ddpm_hierarchical",
#     num_samples=50-
# )

#生成所有氣溫低於等於 25 度(最多1500個樣本)
# error_dataset_low_temp = create_error_dataset(
#     dataset=dataset,
#     diffusion=diffusion,
#     device='cuda',
#     condition_length=8,
#     prediction_length=1,
#     condition_column='氣溫',
#     condition_value=25,
#     operator_str='<=',
#     normalize=True,
#     save_dir=r"C:\\thesis\\code\\result_ddpm_hierarchical",
#     num_samples=None  # 使用所有符合條件的樣本
# )


Generating holiday==1 simulations: 100%|██████████| 1440/1440 [2:45:19<00:00,  6.89s/it] 

Target Errors shape: torch.Size([1440, 1, 1, 21, 21])
Target Errors min: -7.22310209274292, max: 43.4384651184082, mean: 0.05299980565905571, std: 0.8908411860466003


In [12]:
# 手動儲存誤差數據到 CSV
save_error_to_csv(
    target_errors=error_dataset_holiday.target_errors, 
    save_dir=r"C:\\thesis\\code\\result_ddpm_hierarchical",
    condition_column=condition_column,
    operator_str='==',
    condition_value=1,
    num_samples=num_samples 
)

# 產好的可以直接載入
# csv_path = r"C:\thesis\code\result_ddpm_hierarchical\1500_holiday_equals_1_生成資料.csv"
# loaded_error_dataset = load_error_dataset(
#     csv_path=csv_path,
#     dataset=dataset,  # 假設您已定義原始的 dataset
#     condition_length=8,
#     prediction_length=1,
#     normalize=True
# )

# print(f"載入的數據集長度: {len(loaded_error_dataset)}")
# sample = loaded_error_dataset[0]
# print(f"樣本結構: model_input shape={sample[0].shape}, target_error shape={sample[1].shape}, extra_data shape={sample[2].shape}")

INFO:root:目標誤差數據已儲存至 C:\\thesis\\code\\result_ddpm_hierarchical\1440_holiday_equals_1_target_errors.csv


In [13]:
if __name__ == "__main__":
    # 參數設定
    H, W = 21, 21
    condition_length, prediction_length = 8, 1
    batch_size, epochs, lr, timesteps, patience = 144, 144, 0.00144, 1440, 12
    checkpoint_interval = 12
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    save_dir = r"C:\\thesis\\code\\result_ddpm_hierarchical"

    # 數據集
    dataset = PeopleFlowDatasetCondition(
        csv_path=r"C:\\thesis\\code\\Taipei_CF\\all_merged.csv",
        H=H, W=W, condition_length=condition_length, prediction_length=prediction_length,
        normalize=True, debug=True
    )
    train_end = int(0.7 * len(dataset))
    val_end = int(0.85 * len(dataset))
    train_dataset = Subset(dataset, range(0, train_end))
    val_dataset = Subset(dataset, range(train_end, val_end))
    test_dataset = Subset(dataset, range(val_end, len(dataset)))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

訊息: 座標點數量 (495) 多於網格數 (441). 將選擇最靠近地理中心的 441 個座標點進行映射。


In [14]:
# 定義基礎模型
unet = UNet3D(in_channels=1, base_channels=64, time_emb_dim=TIME_EMB_DIM, dropout_rate=0.2).to(device)
diffusion = DDPM3D(model=unet, timesteps=timesteps, beta_start=1e-4, beta_end=0.02, device=device).to(device)
diffusion.extra_columns = dataset.extra_columns  # 確保條件嵌入可用
diffusion.condition_proj = nn.Linear(1, TIME_EMB_DIM)  # 定義條件投影層

# 載入檢查點
checkpoint = torch.load(os.path.join(r"C:\\thesis\\code\\result_ddpm", 'best_model.pth'), map_location=device)

# 使用 strict=False 載入 state_dict，忽略缺少的鍵
diffusion.load_state_dict(checkpoint['model_state_dict'], strict=False)


# 手動初始化 condition_proj 層
# if diffusion.condition_proj is not None:
#     nn.init.xavier_uniform_(diffusion.condition_proj.weight)  # 初始化權重
#     nn.init.zeros_(diffusion.condition_proj.bias)            # 初始化偏差

# 設置已載入的模型
trained_diffusion = diffusion

In [15]:
# 訓練誤差模型
error_dataset = error_dataset_holiday
train_error_end = int(0.7 * len(error_dataset))
val_error_end = int(0.85 * len(error_dataset))
train_error_dataset = Subset(error_dataset, range(0, train_error_end))
val_error_dataset = Subset(error_dataset, range(train_error_end, val_error_end))
test_error_dataset = Subset(error_dataset, range(val_error_end, len(error_dataset)))

train_error_loader = DataLoader(train_error_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_error_loader = DataLoader(val_error_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

error_unet = UNet3D(in_channels=1, base_channels=64, time_emb_dim=TIME_EMB_DIM, dropout_rate=0.2).to(device)
error_diffusion = DDPM3D(
    model=error_unet,
    timesteps=timesteps,
    beta_start=1e-4,
    beta_end=0.02,
    device=device,
).to(device)
error_diffusion.training_dataset = error_dataset

# 設置存檔目錄和模型名稱
error_save_dir = os.path.join(save_dir, 'error_model')
os.makedirs(error_save_dir, exist_ok=True)
model_name = condition_column # 可改為 'weekday' 或其他名稱

# 檢查並載入現有模型
best_model_path = os.path.join(error_save_dir, f'{model_name}_best_model.pth')
checkpoint_path = os.path.join(error_save_dir, f'{model_name}_checkpoint.pth')
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device)
    error_diffusion.load_state_dict(checkpoint['model_state_dict'])
    logging.info(f"已載入最佳模型: {best_model_path}")
elif os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    error_diffusion.load_state_dict(checkpoint['model_state_dict'])
    logging.info(f"已載入檢查點模型: {checkpoint_path}")
else:
    logging.info(f"未找到現有模型: {best_model_path} 或 {checkpoint_path}，將從頭訓練")

INFO:root:已載入最佳模型: C:\\thesis\\code\\result_ddpm_hierarchical\error_model\holiday_best_model.pth


In [16]:
# 訓練誤差模型
for cond, target, extra_data in train_error_loader:
    print(f"cond shape: {cond.shape}")  # 應為 (batch_size, 1, 9, 21, 21)
    print(f"target shape: {target.shape}")  # 應為 (batch_size, 1, 1, 21, 21)
    print(f"extra_data shape: {extra_data.shape}")
    break
trained_error_diffusion = train_ddpm(error_diffusion, train_error_loader, val_error_loader,
                                     model_name=model_name,
                                     epochs=epochs, lr=lr, device=device, patience=patience,
                                     save_dir=error_save_dir,
                                     checkpoint_interval=checkpoint_interval)

cond shape: torch.Size([144, 1, 9, 21, 21])
target shape: torch.Size([144, 1, 1, 21, 21])
extra_data shape: torch.Size([144, 8, 25])
cond shape: torch.Size([144, 1, 9, 21, 21]), target shape: torch.Size([144, 1, 1, 21, 21]), extra_data shape: torch.Size([144, 8, 25])


INFO:root:Epoch [1/144] - Train Loss: 0.1174, Val Loss: 0.1295, Learning Rate: 0.00144000
INFO:root:保存最佳模型，驗證損失: 0.1295, 學習率: 0.00144000
INFO:root:Epoch [2/144] - Train Loss: 0.0845, Val Loss: 0.1036, Learning Rate: 0.00144000
INFO:root:保存最佳模型，驗證損失: 0.1036, 學習率: 0.00144000
INFO:root:Epoch [3/144] - Train Loss: 0.0716, Val Loss: 0.0770, Learning Rate: 0.00144000
INFO:root:保存最佳模型，驗證損失: 0.0770, 學習率: 0.00144000
INFO:root:Epoch [4/144] - Train Loss: 0.0658, Val Loss: 0.0644, Learning Rate: 0.00144000
INFO:root:保存最佳模型，驗證損失: 0.0644, 學習率: 0.00144000
INFO:root:Epoch [5/144] - Train Loss: 0.0603, Val Loss: 0.0704, Learning Rate: 0.00144000
INFO:root:Epoch [6/144] - Train Loss: 0.0600, Val Loss: 0.0470, Learning Rate: 0.00144000
INFO:root:保存最佳模型，驗證損失: 0.0470, 學習率: 0.00144000
INFO:root:Epoch [7/144] - Train Loss: 0.0607, Val Loss: 0.0437, Learning Rate: 0.00144000
INFO:root:保存最佳模型，驗證損失: 0.0437, 學習率: 0.00144000
INFO:root:Epoch [8/144] - Train Loss: 0.0502, Val Loss: 0.0632, Learning Rate: 0.0014400

In [19]:
num_samples = 360
# 評估兩階段模型
metrics_base, metrics_two_stage = evaluate_hierarchical_model(
    trained_diffusion, 
    trained_error_diffusion, 
    test_dataset,
    condition_column=condition_column, 
    condition_value=1, 
    operator_str='==',
    device=device, 
    max_samples=num_samples, 
    save_dir=save_dir
)

# 顯示兩者各指標資訊
logging.info(f"基礎模型 - MSE: {metrics_base['mse']:.6f}, MAE: {metrics_base['mae']:.6f}, "
             f"MAPE: {metrics_base['mape']:.6f}%, SMAPE: {metrics_base['smape']:.6f}%, FID: {metrics_base['fid']:.6f}")
logging.info(f"分層模型 - MSE: {metrics_two_stage['mse']:.6f}, MAE: {metrics_two_stage['mae']:.6f}, "
             f"MAPE: {metrics_two_stage['mape']:.6f}%, SMAPE: {metrics_two_stage['smape']:.6f}%, FID: {metrics_two_stage['fid']:.6f}")

INFO:root:正在從提供的數據集 (長度 2547) 中篩選條件: holiday == 1
INFO:root:在完整數據集中找到 5265 個符合條件的索引。
INFO:root:在輸入數據集子集 (長度 2547) 中找到 797 個符合條件的相對索引。
INFO:root:創建了只包含條件樣本的數據集，長度: 797
INFO:root:將評估 360 個樣本 (從 797 個可用條件樣本中抽取)。
Processing filtered samples: 100%|██████████| 360/360 [44:20<00:00,  7.39s/it]
INFO:root:基礎預測、計算出的誤差模型輸出(原始尺度)和真實值已儲存至 C:\\thesis\\code\\result_ddpm_hierarchical\evaluation_predictions_holiday_==_1.csv
INFO:root:正在計算 360 個樣本的基礎模型和分層模型指標...
INFO:root:指標計算完成。
INFO:root:準備計算 FID...
INFO:root:用於熱力圖正規化的全局範圍 (1%-99%): min=32.00, max=7966.81
INFO:root:計算 Inception mean/std...
INFO:root:計算完成: mean=[0.24213947355747223, 0.26160746812820435, 0.47473883628845215], std=[0.07269380986690521, 0.19817300140857697, 0.07965638488531113]
INFO:root:真實特徵形狀: (360, 2048), 分層模型生成特徵形狀: (360, 2048)
INFO:root:分層模型 FID 計算結果: 188.61635659658845
INFO:root:調用 evaluate_model 評估基礎模型 (在 360 個過濾樣本上)...
INFO:root:正在評估基礎模型，樣本數: 360
INFO:root:evaluate_model 正在從 dataset 中提取前 N 個 target，假設與預計算預測對應。
INFO:root:使用了預計算的 MS